# License Plate Detection and Recognition

## Introduction

In this assignment, you will build a two-stage license plate recognition system:

1. **Stage 1: Detection (LPD)** – Locate and extract license plates from raw images.
2. **Stage 2: Recognition (LPR)** – Identify and classify the individual characters within the detected license plates.

## The Ultralytics Framework

[Ultralytics](https://github.com/ultralytics/ultralytics?utm_source=gemini) is a Python library that streamlines the training, testing, and deployment of YOLO models. You will use it for the detection phase of this project. Key features include:

* **Intuitive API**: A simple Python interface for training, validation, and inference.
* **Pre-trained Models**: Access to various model scales (from nano to extra-large).
* **Export Options**: Native support for exporting to ONNX, TFLite, CoreML, and more.
* **Multi-Task Learning**: Built-in support for object detection, segmentation, and pose estimation.

---

## Assignment Tasks

The project is divided into four distinct phases. Please follow the requirements carefully.

### Phase 1: License Plate Detection (LPD)

* Train a YOLO model on the provided LPD dataset to accurately draw bounding boxes around license plates.
* *Note: You are required to use the Ultralytics framework for this section.*

### Phase 2: License Plate Recognition (LPR)

Train a Convolutional Neural Network (CNN) on the LPR dataset to classify the cropped plates (a **7-digit + 1-letter** classification task).

* *Note: Ultralytics is **strictly prohibited** for this section. You must build this pipeline manually using PyTorch.*

For this phase, you must train and compare **two separate models**:

1. **Custom Architecture (From Scratch):** Design and train your own CNN without using pre-trained weights.
* Incorporate architectural concepts such as Dropout, Batch Normalization, Layer Normalization, and Skip/Residual Connections.


2. **Pre-trained Backbone (Fine-tuning):** Select an appropriate pre-trained model (e.g., MobileNet, EfficientNet) as your backbone, adapt the classification head to this specific task, and fine-tune it.

**Training & Monitoring Requirements:**

* **Hyperparameters:** Configure your optimizer, learning rate scheduler, loss function, weight initialization, activation functions, and early stopping appropriately.
* **Stability:** Ensure vanishing, exploding, or saturating gradients do not occur.
* **Monitoring:** Track and visualize training progress (loss and accuracy curves) and monitor gradient health.
* **Comparison:** Once both models are trained, compare them across key metrics: accuracy, model capacity (parameter count), loss, and inference time.

### Phase 3: End-to-End Pipeline

* Create a unified pipeline that connects Phase 1 and Phase 2. The system should take a raw image, detect the license plate, crop it, and pass it to your best recognition model to output the final string of characters.
* Evaluate the performance of your entire system on the test data using conventional metrics for both detection (e.g., mAP) and classification (e.g., Accuracy, F1-Score).

### Phase 4: Comprehensive Report

* Prepare a detailed report documenting your methodology, architectural choices, training parameters, monitoring graphs, and final comparative results.

---

## Dataset

This project uses a real-world dataset of Iranian vehicle license plates.

**Dataset Link:** [Google Drive Folder](https://drive.google.com/drive/folders/1sFQVSuyiNPbRJT2A5xeSsuZTau8DtF7T?usp=sharing&utm_source=gemini)

The folder contains two archives:

* `LPD.zip`: Images and annotations for License Plate Detection.
* `LPR.zip`: Cropped images and labels for License Plate Recognition.

## 0. Setup

### 0.1 Environment variables (must run before `import torch`)

On Apple Silicon, `PYTORCH_ENABLE_MPS_FALLBACK=1` makes PyTorch run any op the MPS backend doesn't support on the CPU instead of crashing. On Colab and Kaggle the setting has no effect.

In [ ]:
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

In [ ]:
import os, sys, subprocess
import shutil  # CHANGED: needed to check that nvidia-smi exists

print("python:", sys.version.split()[0], "| /content:", os.path.isdir("/content"))
try:
    import google.colab

    print("google.colab: importable")
except ImportError:
    print("google.colab: NOT importable")
import torch

print(
    "cuda:",
    torch.cuda.is_available(),
    "|",
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU",
)
# CHANGED: nvidia-smi does not exist on a Mac (FileNotFoundError), so only call it when it is installed
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
# CHANGED: YOLO26 needs ultralytics>=8.4.0; gdown (imported below) is not preinstalled on every platform
%pip install -q "ultralytics>=8.4.0"
%pip install -q matplotlib opencv-python pyyaml gdown

In [ ]:
import os
import shutil
import time
import random
from tqdm import tqdm

import gdown
from IPython.display import display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data

from torchvision import models
from torchvision import transforms

from ultralytics import YOLO

In [ ]:
# CHANGED: also pick Apple-Silicon "mps" when there is no CUDA GPU (local smoke test on the Mac)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Using device: {device}")

In [ ]:
# Runtime check: is this kernel a Colab runtime, and does it have a GPU?
# Self-contained on purpose (safe to run before the install cell).
import os, platform, subprocess, sys

print("python        :", sys.version.split()[0])
print("platform      :", platform.platform())
print("cwd           :", os.getcwd())
print("/content exists:", os.path.isdir("/content"))
try:
    import google.colab  # noqa: F401

    print("google.colab  : importable (Colab runtime)")
except ImportError:
    print("google.colab  : NOT importable (probably a local kernel)")

try:
    import torch

    print("torch         :", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print("gpu           :", p.name, f"({p.total_memory / 1e9:.1f} GB)")
except ImportError:
    print("torch         : not installed in this kernel")

try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print("nvidia-smi    : not found")

### 0.2 Platform-aware configuration

The next cell holds the only flags you might want to change:

* `SMOKE_TEST`: `None` means automatic. It is `True` on the local Mac (a few dozen images, 1 epoch, tiny image size) and `False` on Colab/Kaggle (full training). You can also pass it with papermill: `papermill ... -p SMOKE_TEST True`.
* `USE_DRIVE` (Colab only): set it to `True` to mount Google Drive and keep `runs/` (checkpoints) there, so a disconnected session can resume. The final `lpd_lpr_artifacts.zip` is copied to Drive as well.

The platform is detected automatically (Colab, Kaggle or local). Every path used later comes from the variables defined here.

In [ ]:
# Parameters (papermill overrides these)
SMOKE_TEST = None  # None = auto: True on the local Mac, False on Colab / Kaggle
USE_DRIVE = False  # Colab only: keep checkpoints + artifacts on Google Drive

In [ ]:
import contextlib
import copy
import hashlib
import importlib
import json
import re
import warnings
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import ultralytics
import yaml
from packaging.version import Version


# ---------------------------------------------------------------- platform detection
def _in_colab():
    if not os.path.isdir("/content"):
        return False
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


# Kaggle is checked FIRST: Kaggle's current image also has /content and an importable google.colab,
# which made the first Kaggle run write everything to /content (not saved as notebook output).
IS_KAGGLE = os.path.isdir("/kaggle/working") or any(k in os.environ for k in ("KAGGLE_KERNEL_RUN_TYPE", "KAGGLE_URL_BASE", "KAGGLE_DATA_PROXY_TOKEN"))
IS_COLAB = not IS_KAGGLE and _in_colab()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)
PLATFORM = "colab" if IS_COLAB else "kaggle" if IS_KAGGLE else "local"

if SMOKE_TEST is None:
    SMOKE_TEST = IS_LOCAL or os.environ.get("SMOKE_TEST", "").lower() in ("1", "true", "yes")
SMOKE_TEST = str(SMOKE_TEST).lower() in ("1", "true", "yes")  # papermill may pass a string
RUN_TAG = "smoke" if SMOKE_TEST else "full"

# ---------------------------------------------------------------- paths
if IS_COLAB:
    WORK_DIR = Path("/content")
elif IS_KAGGLE:
    WORK_DIR = Path("/kaggle/working")
else:  # the notebook's own folder (Jupyter / papermill run with the notebook folder as cwd)
    WORK_DIR = Path(os.environ.get("LPR_NOTEBOOK_DIR", os.getcwd())).resolve()



def _pick_data_root():
    """Kaggle saves everything in /kaggle/working as notebook output, so the ~4 GB of zips + extracted
    images go to non-persisted scratch disk there (if it has room)."""
    if IS_KAGGLE:
        for cand in (Path("/kaggle/tmp"), Path("/tmp/lpd_lpr_data")):
            try:
                cand.mkdir(parents=True, exist_ok=True)
                if shutil.disk_usage(cand).free > 12e9:
                    return cand
            except OSError:
                pass
    return WORK_DIR


DATA_ROOT = _pick_data_root()  # where cloud downloads / extraction happen (the notebook folder locally)
LPD_DIR = DATA_ROOT / "LPD"  # read-only input: images/, labels/, plate_labels.csv
LPR_DIR = DATA_ROOT / "LPR"  # read-only input: detections/, valid_samples.csv
OUT_DIR = WORK_DIR / "work" if IS_LOCAL else WORK_DIR  # everything this notebook writes
PERSIST_DIR = OUT_DIR
if IS_COLAB and USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    PERSIST_DIR = Path("/content/drive/MyDrive/lpd_lpr")
RUNS_DIR = PERSIST_DIR / "runs"  # YOLO + LPR checkpoints (on Drive when USE_DRIVE)
ARTIFACT_DIR = OUT_DIR / "lpd_lpr_artifacts"  # Phase-3 bundle
LPD_YOLO_DIR = (OUT_DIR if IS_LOCAL else DATA_ROOT) / "lpd_yolo"  # Ultralytics layout (symlinks + label copies)
WEIGHTS_DIR = OUT_DIR / "weights"  # downloaded pretrained YOLO weights
LPR_MODULE_DIR = OUT_DIR / "lpr_module"  # generated lpr_models.py
MODELS_DIR = OUT_DIR / "models"  # one output folder per trained model, written right after its training
RESULTS_DIR = OUT_DIR / "outputs"  # CSV result tables + text run log (/kaggle/working/outputs on Kaggle)
FIG_DIR = RESULTS_DIR / "figures"  # every figure shown in this notebook, saved as a 150-dpi PNG
for _d in (OUT_DIR, RUNS_DIR, ARTIFACT_DIR, WEIGHTS_DIR, LPR_MODULE_DIR / "parts", FIG_DIR, MODELS_DIR):
    _d.mkdir(parents=True, exist_ok=True)
RUN_LOG = RESULTS_DIR / f"run_log_{RUN_TAG}.txt"

# one figure per plot, large and readable (also in the saved PNGs)
plt.rcParams.update({"figure.figsize": (11, 6.5), "figure.dpi": 100, "savefig.dpi": 150, "font.size": 13,
                     "axes.titlesize": 16, "axes.labelsize": 14, "xtick.labelsize": 12, "ytick.labelsize": 12,
                     "legend.fontsize": 12, "axes.grid": True, "grid.alpha": 0.3})


def log(*args):
    """print() + append a timestamped line to RUN_LOG (the text log of this run)."""
    msg = " ".join(str(a) for a in args)
    print(msg)
    with open(RUN_LOG, "a", encoding="utf-8") as f:
        f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')}  {msg}\n")


def show(name):
    """Save the current matplotlib figure to FIG_DIR/<name>.png, then display it in the notebook."""
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")
    plt.show()
    plt.close("all")


def free_gpu():
    """Release cached GPU memory between the big steps (YOLO -> analysis -> LPR)."""
    import gc

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ---------------------------------------------------------------- device / loader settings
DEVICE = device
IS_CUDA = DEVICE.type == "cuda"
IS_MPS = DEVICE.type == "mps"
N_GPUS = torch.cuda.device_count() if IS_CUDA else 0
# LPR models on one GPU: they are small, data loading (CPU) is the bottleneck, and nn.DataParallel made the
# frozen-backbone stage ~8x slower before the kernel died on Kaggle T4 x2. YOLO still trains on all GPUs (DDP).
LPR_MULTI_GPU = False
YOLO_DEVICE = 0 if IS_CUDA else ("mps" if IS_MPS else "cpu")  # predict / val (single device)
YOLO_TRAIN_DEVICE = list(range(N_GPUS)) if N_GPUS > 1 else YOLO_DEVICE  # e.g. [0, 1] on Kaggle T4 x2 -> DDP
NUM_WORKERS = 0 if IS_LOCAL else min(2 if IS_COLAB else 4, os.cpu_count() or 1)
PIN_MEMORY = IS_CUDA
USE_AMP = IS_CUDA  # fp16 autocast + GradScaler only on CUDA; fp32 on mps / cpu

if IS_CUDA:
    _gpu = torch.cuda.get_device_name(0)
    for _i in range(N_GPUS):
        print(f"GPU {_i}: {torch.cuda.get_device_name(_i)} ({torch.cuda.get_device_properties(_i).total_memory / 1e9:.1f} GB)")
    if "P100" in _gpu:
        warnings.warn(
            "Kaggle assigned a Tesla P100: recent PyTorch builds no longer ship kernels for its "
            "compute capability (6.0) and may fail or run slowly. Switch the accelerator to 'GPU T4 x2'."
        )


# ---------------------------------------------------------------- reproducibility
SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if IS_MPS:
        torch.mps.manual_seed(seed)


set_seed()

# ---------------------------------------------------------------- ultralytics version (YOLO26 needs >= 8.4.0)
print("ultralytics", ultralytics.__version__)
assert Version(ultralytics.__version__) >= Version("8.4.0"), "YOLO26 needs ultralytics>=8.4.0; re-run the %pip cell and restart"

# ---------------------------------------------------------------- one config dict for every training cell
CFG = {
    "seed": SEED,
    "lpd": {
        "model": "yolo26n.pt",  # try "yolo26s.pt" (~2.5x params) if mAP50-95 plateaus
        "split": {"train": 0.70, "val": 0.15, "test": 0.15},
        "smoke_n": {"train": 48, "val": 16, "test": 40},
        "train": {
            "epochs": 40,
            "imgsz": 960,  # plates are small (median ~20 px high at native res), so 640 loses detail
            "batch": 16,
            "patience": 15,
            "optimizer": "MuSGD",  # YOLO26 default; explicit because "auto" would switch to AdamW below 10k iterations (40 ep on 2 GPUs ~5.5k)
            "lr0": 0.01,
            "cos_lr": True,
            "save_period": 10,  # extra epochN.pt checkpoints besides last.pt/best.pt
            "close_mosaic": 10,
            "mosaic": 1.0,
            "fliplr": 0.5,
            "degrees": 0.0,
            "translate": 0.1,
            "scale": 0.5,
            "hsv_h": 0.015,
            "hsv_s": 0.7,
            "hsv_v": 0.4,
            "plots": True,
        },
    },
    "lpr": {
        "split": {"holdout": 0.20},  # of the plate groups not forced into test; halved into val / test
        "smoke_n": {"train": 256, "val": 64, "test": 64},
        "batch_size": 128,
        "label_smoothing": 0.05,
        "grad_clip": 5.0,
        "pct_start": 0.15,  # OneCycle warm-up fraction
        "letter_weight_power": 0.5,  # letter class weight = (N / count) ** power
        "max_letter_weight": 5.0,
        "crop_jitter": 0.08,  # random crop / pad up to 8% per side to mimic loose/tight YOLO boxes
        "log_every": 10,  # steps between gradient-health samples
        "scratch": {"epochs": 40, "lr": 2e-3, "weight_decay": 0.05, "patience": 8},
        "pretrained": {
            "backbone": "mobilenet_v3_large",
            "stage1_epochs": 3,
            "stage1_lr": 1e-3,
            "epochs": 25,
            "head_lr": 1e-3,
            "backbone_lr": 3e-4,
            "weight_decay": 0.01,
            "patience": 6,
        },
        "timing": {"warmup": 10, "iters": 50, "batch": 64},
    },
}
SMOKE_OVERRIDES = {
    "lpd": {"train": {"epochs": 1, "imgsz": 320, "batch": 8, "save_period": 1, "close_mosaic": 0}},
    "lpr": {
        "batch_size": 32,
        "log_every": 1,
        "scratch": {"epochs": 1},
        "pretrained": {"stage1_epochs": 1, "epochs": 1},
        "timing": {"warmup": 2, "iters": 3, "batch": 16},
    },
}


def _deep_update(d, u):
    for k, v in u.items():
        if isinstance(v, dict):
            _deep_update(d[k], v)
        else:
            d[k] = v


if SMOKE_TEST:
    _deep_update(CFG, SMOKE_OVERRIDES)

log(f"===== run started: platform={PLATFORM}  smoke_test={SMOKE_TEST}  device={DEVICE}  gpus={N_GPUS}  yolo_train_device={YOLO_TRAIN_DEVICE}")
print(f"workers={NUM_WORKERS}  pin_memory={PIN_MEMORY}  amp={USE_AMP}")
for _n in ("WORK_DIR", "DATA_ROOT", "MODELS_DIR", "LPD_DIR", "LPR_DIR", "OUT_DIR", "RUNS_DIR", "ARTIFACT_DIR", "LPD_YOLO_DIR"):
    print(f"{_n:13s} = {globals()[_n]}")
print(json.dumps(CFG, indent=1))

### 0.3 Data sources and shared plate-text helpers

The notebook finds the data in this order:

1. **Local (Mac):** it uses `./LPD` and `./LPR` as they are. Nothing is downloaded, unzipped, or written into those folders.
2. **Kaggle:** it searches `/kaggle/input/` for an attached dataset, either already-extracted `LPD`/`LPR` folders or `LPD.zip`/`LPR.zip`, and links it into `/kaggle/working`.
3. **Otherwise** it uses the original `gdown` cells. They only run when the data is still missing, so re-running them is harmless.

> **Kaggle:** turn on *Settings → Internet → On*. It is needed for `gdown`, for `pip install ultralytics`, and for the pretrained YOLO26 / MobileNetV3 weights.

After extraction, `LPD_DIR` / `LPR_DIR` always have the same layout as the local folders (`images/ labels/ plate_labels.csv` and `detections/ valid_samples.csv`). The layout is detected whether or not the zip has a top-level folder.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
DATA_DIRS = {"LPD": LPD_DIR, "LPR": LPR_DIR}
DATA_ZIPS = {"LPD": DATA_ROOT / "LPD.zip", "LPR": DATA_ROOT / "LPR.zip"}
EXTRACT_DIRS = {"LPD": DATA_ROOT / "LPD_FILES", "LPR": DATA_ROOT / "LPR_FILES"}
LPD_ZIP, LPR_ZIP = DATA_ZIPS["LPD"], DATA_ZIPS["LPR"]
LPD_EXTRACT_DIR, LPR_EXTRACT_DIR = EXTRACT_DIRS["LPD"], EXTRACT_DIRS["LPR"]


def is_dataset_root(p, kind):
    p = Path(p)
    if kind == "LPD":
        return (p / "images").is_dir() and (p / "labels").is_dir()
    return (p / "detections").is_dir() and (p / "valid_samples.csv").is_file()


def find_dataset_root(base, kind, max_depth=4):
    """Breadth-first search for the folder that has the LPD / LPR layout (zip with or without top-level folder)."""
    level = [Path(base)] if Path(base).is_dir() else []
    for _ in range(max_depth + 1):
        nxt = []
        for p in level:
            if is_dataset_root(p, kind):
                return p
            try:
                nxt += sorted(c for c in p.iterdir() if c.is_dir() and not c.name.startswith(("__MACOSX", ".")))
            except (PermissionError, OSError):
                pass
        level = nxt
    return None


def link_dataset(src, dst):
    """Make dst (LPD_DIR / LPR_DIR) point at src with a symlink; src may be read-only (/kaggle/input)."""
    src, dst = Path(src).resolve(), Path(dst)
    if dst.is_symlink() and not dst.exists():
        dst.unlink()  # dangling link from an earlier session
    if is_dataset_root(dst, "LPD" if dst == LPD_DIR else "LPR"):
        return
    if dst.exists():
        if dst.is_dir() and not any(dst.iterdir()):
            dst.rmdir()
        else:
            raise RuntimeError(f"{dst} exists but does not look like the dataset; remove it and re-run")
    if src != dst.resolve():
        dst.symlink_to(src, target_is_directory=True)


DATA_SOURCE = {}
for _kind, _dir in DATA_DIRS.items():
    if IS_LOCAL:
        assert is_dataset_root(_dir, _kind), f"expected the extracted {_kind} dataset at {_dir}"
        DATA_SOURCE[_kind] = f"local folder {_dir}"
    elif is_dataset_root(_dir, _kind):
        DATA_SOURCE[_kind] = f"already prepared at {_dir}"
    else:
        DATA_SOURCE[_kind] = "gdown (Google Drive)"
        if IS_KAGGLE and Path("/kaggle/input").is_dir():
            _root = find_dataset_root("/kaggle/input", _kind, max_depth=5)
            if _root is not None:
                link_dataset(_root, _dir)
                DATA_SOURCE[_kind] = f"Kaggle input {_root}"
            else:
                _zips = sorted(Path("/kaggle/input").rglob(f"{_kind}.zip"))
                if _zips:
                    DATA_ZIPS[_kind] = _zips[0]
                    DATA_SOURCE[_kind] = f"Kaggle input zip {_zips[0]}"
LPD_ZIP, LPR_ZIP = DATA_ZIPS["LPD"], DATA_ZIPS["LPR"]


def need_download(kind):
    return not IS_LOCAL and not is_dataset_root(DATA_DIRS[kind], kind) and not DATA_ZIPS[kind].exists()


def need_unzip(kind):
    return not IS_LOCAL and not is_dataset_root(DATA_DIRS[kind], kind)


def finalize_extraction(kind):
    root = find_dataset_root(EXTRACT_DIRS[kind], kind)
    assert root is not None, f"could not find the {kind} layout inside {EXTRACT_DIRS[kind]}"
    link_dataset(root, DATA_DIRS[kind])
    print(f"{kind}: {DATA_DIRS[kind]} -> {root}")
    if DATA_ZIPS[kind].parent == DATA_ROOT and DATA_ZIPS[kind].exists():
        DATA_ZIPS[kind].unlink()  # our own download: free the disk space (never touches /kaggle/input)


for _kind, _src in DATA_SOURCE.items():
    print(f"{_kind}: {_src}")

# ---------------------------------------------------------------- plate text helpers
# Letters in Persian alphabet order with a Latin transliteration (matplotlib can't shape Persian script,
# so plots use the Latin form). "الف" is stored spelled out (3 code points) in the data but is ONE letter.
LETTER_LATIN = {
    "الف": "Alef", "آ": "A", "ب": "B", "پ": "P", "ت": "T", "ث": "Se", "ج": "J", "چ": "Ch", "ح": "He",
    "خ": "Kh", "د": "D", "ذ": "Zal", "ر": "R", "ز": "Z", "ژ": "Zh", "س": "S", "ش": "Sh", "ص": "Sad",
    "ض": "Zad", "ط": "Ta", "ظ": "Za", "ع": "Ein", "غ": "Ghein", "ف": "F", "ق": "Gh", "ک": "K", "گ": "G",
    "ل": "L", "م": "M", "ن": "N", "و": "V", "ه": "H", "ی": "Y",
}
_DIGIT_TRANS = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")
_PLATE_RE = re.compile(r"^(\d{2})(\D+)(\d{5})$")


def normalize_plate(text):
    """Canonical plate string: ASCII digits, no ZWJ/ZWNJ/spaces, Persian (not Arabic) yeh/kaf."""
    t = str(text).strip().translate(_DIGIT_TRANS)
    for ch in ("‍", "‌", " ", "‏", "‎"):
        t = t.replace(ch, "")
    return t.replace("ي", "ی").replace("ك", "ک")


def parse_plate(text):
    """'12ب34567' -> ('1234567', 'ب');  None if it is not 2 digits + known letter + 5 digits."""
    m = _PLATE_RE.match(normalize_plate(text))
    if not m or m.group(2) not in LETTER_LATIN:
        return None
    return m.group(1) + m.group(3), m.group(2)


def plate_to_latin(text):
    p = parse_plate(text)
    if p is None:
        return "?" + normalize_plate(text).encode("ascii", "replace").decode()
    d, letter = p
    return f"{d[:2]}-{LETTER_LATIN[letter]}-{d[2:5]}-{d[5:]}"

### Models trained in this notebook

| phase | model | job | starting point | size |
|---|---|---|---|---|
| 1: detection | **YOLO26n** (Ultralytics) | find plate boxes in the full car photo | COCO-pretrained `yolo26n.pt`, fine-tuned ≤ 40 epochs at 960 px | 2.4 M params |
| 2: recognition, model I | **ScratchLPRNet** (own residual CNN) | read the 8 characters of a cropped plate | random init (Kaiming), ≤ 40 epochs | 7.4 M params |
| 2: recognition, model II | **MobileNetV3-Large + the same head** | same job, pretrained backbone | ImageNet weights: 3 epochs frozen, then ≤ 25 epochs fine-tuning | 4.3 M params |

The recognizer with the higher test full-plate accuracy goes into the Phase-3 bundle. Alternatives you can switch on in `CFG` (cell 0.2): `yolo26s.pt` for the detector, `efficientnet_b0` for the pretrained backbone.

**Where the models go:** each model gets its own output folder **as soon as its training finishes**, so a later error can't lose it:
`models/1_lpd_yolo26n/`, `models/2_lpr_scratch/`, `models/3_lpr_pretrained/` (on Kaggle under `/kaggle/working/models/`).

**Where the results go**
* Every figure is shown in the notebook **and** saved as a large PNG (150 dpi) to `outputs/figures/` (on Kaggle: `/kaggle/working/outputs/figures/`).
* Result tables are saved as CSV in `outputs/`.
* Key events and metrics are appended to the text log `outputs/run_log_<smoke|full>.txt`.
* The same files are also packed into `lpd_lpr_artifacts.zip` (under `outputs/`). The last cell lists every file.

## 1. License Plate Detection - YOLO

### Data Load and Split

In [ ]:
# CHANGED: skipped when the data is already there (local ./LPD, Kaggle input, or an earlier run);
#          -O saves the zip to DATA_ROOT instead of the current directory.
if need_download("LPD"):
    !gdown --fuzzy https://drive.google.com/file/d/1UBV930A9pRhVwop-WunmP38th2XwEumx/view?usp=drive_link -O "{LPD_ZIP}"
else:
    print("LPD download skipped:", DATA_SOURCE["LPD"])

In [ ]:
# CHANGED: the original `!mkdir /content/LPD_FILES` + `!unzip LPD.zip -d /content/LPD_FILES` hard-coded
#          Colab paths and failed on re-run. Now: skipped when the data is already there, idempotent
#          (mkdir -p, unzip -o), and LPD_DIR is linked to whatever folder the zip contains.
if need_unzip("LPD"):
    !mkdir -p "{LPD_EXTRACT_DIR}"
    !unzip -q -o "{LPD_ZIP}" -d "{LPD_EXTRACT_DIR}"
    finalize_extraction("LPD")
else:
    print("LPD unzip skipped:", DATA_SOURCE["LPD"])

### 1.1 Inspect the LPD data

This step reads the dataset without changing anything. It checks the folder layout, file formats, annotation format, classes, and whether the plate text (the character string) is annotated.

In [ ]:
print("LPD_DIR:", LPD_DIR)
for p in sorted(LPD_DIR.iterdir()):
    if not p.name.startswith("."):
        print(f"  {p.name:18s}", "dir" if p.is_dir() else f"{p.stat().st_size / 1e3:.0f} kB")

lpd_images = {p.stem: p for p in sorted((LPD_DIR / "images").iterdir()) if p.suffix.lower() in IMG_EXTS}
lpd_labels = {p.stem: p for p in sorted((LPD_DIR / "labels").glob("*.txt"))}
print("image extensions:", Counter(p.suffix.lower() for p in lpd_images.values()))
print(f"{len(lpd_images)} images, {len(lpd_labels)} label files")
print("images without label file:", len(set(lpd_images) - set(lpd_labels)),
      "| label files without image:", len(set(lpd_labels) - set(lpd_images)))

_ex = next(iter(lpd_labels.values()))
print(f"\nexample label {_ex.name}:\n{_ex.read_text().strip()}")
_cls = Counter(l.split()[0] for p in lpd_labels.values() for l in p.read_text().splitlines() if l.strip())
_ncol = Counter(len(l.split()) for p in lpd_labels.values() for l in p.read_text().splitlines() if l.strip())
print("class ids:", dict(_cls), "| columns per line:", dict(_ncol))
print("=> annotation format: YOLO txt, one line per plate: `class cx cy w h` (normalized); single class 0 = license plate")

# Plate text?
LPD_PLATE_CSV = LPD_DIR / "plate_labels.csv"
LPD_HAS_PLATE_TEXT = LPD_PLATE_CSV.is_file()
LPD_PLATES = {}
if LPD_HAS_PLATE_TEXT:
    _pdf = pd.read_csv(LPD_PLATE_CSV, dtype=str, encoding="utf-8", keep_default_na=False)
    display(_pdf.head())
    for r in _pdf.itertuples():
        LPD_PLATES[Path(r.image_path).stem] = [normalize_plate(t) for t in r.label.split()]
    _nbox = {s: sum(1 for l in p.read_text().splitlines() if l.strip()) for s, p in lpd_labels.items()}
    _agree = sum(len(LPD_PLATES.get(s, [])) == n for s, n in _nbox.items())
    _parsable = sum(parse_plate(t) is not None for ts in LPD_PLATES.values() for t in ts)
    print(f"plate_labels.csv: {len(_pdf)} rows; #plate strings == #boxes for {_agree}/{len(_nbox)} images; "
          f"{_parsable}/{sum(map(len, LPD_PLATES.values()))} strings parse as 2 digits + letter + 5 digits")
print(f"\nLPD annotations include plate text: {LPD_HAS_PLATE_TEXT} "
      "(plate_labels.csv; several plates in one image are space-separated. Their order vs. the box order "
      "in the .txt file is not documented, so Phase 3 should match predicted strings to boxes per image)")

### 1.2 LPD exploratory data analysis and sanity checks

Broken images, missing or empty labels, malformed lines, and boxes outside the image are **reported and skipped**, never modified. Boxes that overshoot the border by less than 1% of the image are clipped. That kind of overshoot is ordinary annotation noise.

In [ ]:
LPD_BAD = defaultdict(list)
LPD_CLEAN_LABELS = {}  # stem -> cleaned YOLO label text
_img_rows, _box_rows = [], []
for stem, ip in tqdm(lpd_images.items(), desc="scan LPD", mininterval=10):
    try:
        with Image.open(ip) as im:
            w, h = im.size
            im.verify()
    except Exception:
        LPD_BAD["broken_image"].append(stem)
        continue
    lp = lpd_labels.get(stem)
    if lp is None:
        LPD_BAD["missing_label"].append(stem)
        continue
    lines = [l.split() for l in lp.read_text().splitlines() if l.strip()]
    if not lines:
        LPD_BAD["empty_label"].append(stem)
        continue
    boxes, problem = [], None
    for parts in lines:
        try:
            assert len(parts) == 5 and int(parts[0]) == 0
            cx, cy, bw, bh = map(float, parts[1:])
        except (AssertionError, ValueError):
            problem = "malformed_line"
            break
        x1, y1, x2, y2 = cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2
        if bw <= 0 or bh <= 0 or min(x1, y1) < -0.01 or max(x2, y2) > 1.01:
            problem = "out_of_range"
            break
        x1, y1, x2, y2 = max(x1, 0.0), max(y1, 0.0), min(x2, 1.0), min(y2, 1.0)  # clip tiny overshoot
        boxes.append(((x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1))
    if problem:
        LPD_BAD[problem].append(stem)
        continue
    LPD_CLEAN_LABELS[stem] = "".join(f"0 {a:.6f} {b:.6f} {c:.6f} {d:.6f}\n" for a, b, c, d in boxes)
    _img_rows.append({"stem": stem, "w": w, "h": h, "n_boxes": len(boxes)})
    _box_rows += [{"stem": stem, "cx": a, "cy": b, "bw": c, "bh": d, "w_px": c * w, "h_px": d * h} for a, b, c, d in boxes]

LPD_VALID = sorted(LPD_CLEAN_LABELS)
lpd_img_df, lpd_box_df = pd.DataFrame(_img_rows), pd.DataFrame(_box_rows)
lpd_box_df["aspect"] = lpd_box_df.w_px / lpd_box_df.h_px
print(f"valid images: {len(LPD_VALID)} / {len(lpd_images)}   boxes: {len(lpd_box_df)}")
print("skipped:", {k: len(v) for k, v in LPD_BAD.items()} or "none", {k: v[:5] for k, v in LPD_BAD.items()})
print("\nimage sizes (top 8):", Counter(zip(lpd_img_df.w, lpd_img_df.h)).most_common(8))
print("boxes per image:", dict(sorted(Counter(lpd_img_df.n_boxes).items())))
display(lpd_box_df[["w_px", "h_px", "aspect", "bw", "bh"]].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(3))

plt.figure()
plt.scatter(lpd_img_df.w, lpd_img_df.h, s=8, alpha=0.3)
plt.title("LPD image sizes")
plt.xlabel("width (px)")
plt.ylabel("height (px)")
show("lpd_image_sizes")

_bc = Counter(lpd_img_df.n_boxes)
plt.figure()
plt.bar(list(_bc), list(_bc.values()))
for k, v in _bc.items():
    plt.text(k, v, str(v), ha="center", va="bottom")
plt.yscale("log")
plt.title("number of plates per image")
plt.xlabel("plates in the image")
plt.ylabel("images (log scale)")
show("lpd_plates_per_image")

plt.figure()
plt.hist(lpd_box_df.h_px, bins=60)
plt.axvline(lpd_box_df.h_px.median(), color="r", ls="--", label=f"median {lpd_box_df.h_px.median():.0f} px")
plt.title("plate box height (native resolution)")
plt.xlabel("height (px)")
plt.ylabel("plates")
plt.legend()
show("lpd_box_height")

plt.figure()
plt.hist(lpd_box_df.w_px, bins=60)
plt.axvline(lpd_box_df.w_px.median(), color="r", ls="--", label=f"median {lpd_box_df.w_px.median():.0f} px")
plt.title("plate box width (native resolution)")
plt.xlabel("width (px)")
plt.ylabel("plates")
plt.legend()
show("lpd_box_width")

plt.figure()
plt.hist(lpd_box_df.aspect.clip(0, 12), bins=60)
plt.title("plate box aspect ratio (width / height)")
plt.xlabel("w / h")
plt.ylabel("plates")
show("lpd_box_aspect_ratio")

plt.figure(figsize=(9, 8))
_hm = plt.hist2d(lpd_box_df.cx, lpd_box_df.cy, bins=40, range=[[0, 1], [0, 1]], cmap="viridis")
plt.gca().invert_yaxis()
plt.colorbar(_hm[3], label="plates")
plt.title("where plates are in the image (box centres)")
plt.xlabel("x (normalized)")
plt.ylabel("y (normalized)")
plt.grid(False)
show("lpd_box_centre_heatmap")

_rng = random.Random(SEED)
fig, axes = plt.subplots(2, 3, figsize=(20, 13))
for a, stem in zip(axes.ravel(), _rng.sample(LPD_VALID, 6)):
    img = cv2.cvtColor(cv2.imread(str(lpd_images[stem])), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    for l in LPD_CLEAN_LABELS[stem].splitlines():
        _, cx, cy, bw, bh = map(float, l.split())
        cv2.rectangle(img, (int((cx - bw / 2) * W), int((cy - bh / 2) * H)), (int((cx + bw / 2) * W), int((cy + bh / 2) * H)), (0, 255, 0), max(3, W // 250))
    a.imshow(img)
    a.set_title(f"{stem}: " + ", ".join(plate_to_latin(t) for t in LPD_PLATES.get(stem, [])), fontsize=13)
    a.axis("off")
plt.suptitle("LPD samples with ground-truth boxes", fontsize=18)
show("lpd_samples")

### 1.3 Data preparation (Ultralytics layout and 70/15/15 split)

* Images that share a plate string (the same vehicle photographed more than once) are kept in the **same split**, so the test set really is unseen.
* The split uses a fixed seed and is roughly 70/15/15. **The test split is held out**: it is never used for training or model selection. Phase 3 evaluates on it, and its file list is saved to `ARTIFACT_DIR/lpd_test_split.txt`.
* `./LPD` stays read-only. The Ultralytics layout is created in `LPD_YOLO_DIR`: `images/{train,val,test}` are **symlinks** to the original JPGs, and `labels/{train,val,test}` hold the cleaned YOLO txt files.

In [ ]:
# TODO (done): Prepare the data for training Ultralytics YOLO model.
# Read the document of Ultralytics.
# It takes the data in special format and directories.

# 1) group images that share a plate string (union-find), then split whole groups
_parent = {s: s for s in LPD_VALID}


def _find(s):
    while _parent[s] != s:
        _parent[s] = _parent[_parent[s]]
        s = _parent[s]
    return s


_first = {}
for s in LPD_VALID:
    for t in LPD_PLATES.get(s, []):
        if t in _first:
            _parent[_find(s)] = _find(_first[t])
        else:
            _first[t] = s
_groups = defaultdict(list)
for s in LPD_VALID:
    _groups[_find(s)].append(s)
_group_list = sorted((sorted(g) for g in _groups.values()), key=lambda g: g[0])
random.Random(SEED).shuffle(_group_list)

_fr, _n = CFG["lpd"]["split"], len(LPD_VALID)
LPD_SPLITS = {"train": [], "val": [], "test": []}
for g in _group_list:
    if len(LPD_SPLITS["test"]) < _fr["test"] * _n:
        LPD_SPLITS["test"] += g
    elif len(LPD_SPLITS["val"]) < _fr["val"] * _n:
        LPD_SPLITS["val"] += g
    else:
        LPD_SPLITS["train"] += g
LPD_SPLITS = {k: sorted(v) for k, v in LPD_SPLITS.items()}
assert not (set(LPD_SPLITS["train"]) & set(LPD_SPLITS["test"])) and not (set(LPD_SPLITS["val"]) & set(LPD_SPLITS["test"]))
log(f"LPD split: {len(_groups)} vehicle groups over {_n} images;",
      {k: f"{len(v)} ({len(v) / _n:.1%})" for k, v in LPD_SPLITS.items()})
for k, v in LPD_SPLITS.items():  # full split lists (the smoke subset below is only for training speed)
    (ARTIFACT_DIR / f"lpd_{k}_split.txt").write_text("".join(f"{lpd_images[s].name}\n" for s in v))

# 2) smoke test: only a few dozen images per split
LPD_USE = {k: v[: CFG["lpd"]["smoke_n"][k]] if SMOKE_TEST else v for k, v in LPD_SPLITS.items()}

# 3) build the YOLO layout (rebuilt only when the split / subset changes -> idempotent)
assert LPD_DIR.resolve() not in LPD_YOLO_DIR.resolve().parents, "never write into the raw LPD folder"
_sig = hashlib.md5(json.dumps(LPD_USE, sort_keys=True).encode()).hexdigest()
_marker = LPD_YOLO_DIR / ".split_signature"
if not _marker.exists() or _marker.read_text() != _sig:
    if LPD_YOLO_DIR.exists():
        shutil.rmtree(LPD_YOLO_DIR)  # only symlinks + label copies live here
    for split, stems in LPD_USE.items():
        (LPD_YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (LPD_YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)
        for s in stems:
            src = lpd_images[s].resolve()
            dst = LPD_YOLO_DIR / "images" / split / src.name
            try:
                dst.symlink_to(src)
            except OSError:
                shutil.copy2(src, dst)
            (LPD_YOLO_DIR / "labels" / split / f"{s}.txt").write_text(LPD_CLEAN_LABELS[s])
    _marker.write_text(_sig)
    print("built", LPD_YOLO_DIR)
else:
    print("YOLO layout up to date:", LPD_YOLO_DIR)
for split in LPD_USE:
    print(f"  {split:5s}: {len(list((LPD_YOLO_DIR / 'images' / split).iterdir()))} images")

### Configuration & Model Setup & Train

Define training hyperparameters, specify the dataset configuration path, and load the pretrained YOLO weights. Create this file and put it in the /content directory of Google Colab.

The yaml file (data.yaml) must have the following structure:
```
# Root directory of your dataset
path: /content/LPD_FILES

# Relative paths to image folders (relative to 'path' above)
train: images/train
val: images/val

# Number of classes and mapping from class ID to name
nc: 1
names:
  0: license_plate
```

### 1.4 `data.yaml` and hyperparameters

With 2 GPUs (Kaggle T4 x2), YOLO trains on both with DDP (`device=[0, 1]`), and the batch is multiplied by the GPU count so each GPU keeps a batch of 16. If DDP fails, training retries on one GPU. Validation and prediction use GPU 0.

The file follows the template above, with `path` pointing at the prepared layout `LPD_YOLO_DIR` and a `test: images/test` line added. It is written to `OUT_DIR/data.yaml`, which is `/content/data.yaml` on Colab. All hyperparameters come from `CFG["lpd"]` (cell 0.2).

YOLO26 specifics:
* It needs `ultralytics>=8.4.0`; the version is asserted in 0.2.
* The optimizer is set explicitly to **MuSGD** (lr0 = 0.01), YOLO26's default. With `optimizer="auto"`, Ultralytics picks MuSGD only above 10k iterations. The 40-epoch run is about 11k iterations on one GPU (batch 16) but only about 5.5k on two GPUs (batch 32), so "auto" would silently switch to AdamW on Kaggle's T4 x2.
* It is NMS-free (end-to-end head) and has no DFL loss, so no NMS/IoU settings are tuned and loss columns are detected by name.

In [ ]:
# TODO (done): Setup the configuration file and YOLO model from Ultralytics.
# Feel free to test different YOLO models.
# Define the hyperparameters of training and train the model.
LPD_CFG = CFG["lpd"]
LPD_DATA_YAML = OUT_DIR / "data.yaml"
LPD_DATA_YAML.write_text(
    f"""# Root directory of your dataset
path: {json.dumps(str(LPD_YOLO_DIR))}

# Relative paths to image folders (relative to 'path' above)
train: images/train
val: images/val
test: images/test

# Number of classes and mapping from class ID to name
nc: 1
names:
  0: license_plate
"""
)
print(LPD_DATA_YAML.read_text())
print("model:", LPD_CFG["model"])
print(json.dumps(LPD_CFG["train"], indent=1))

### 1.5 Download the pretrained YOLO26 weights

In [ ]:
# TODO (done): Downloading model
# The official COCO-pretrained yolo26n.pt is downloaded into WEIGHTS_DIR on first use (then reused).
LPD_WEIGHTS = WEIGHTS_DIR / LPD_CFG["model"]
lpd_model = YOLO(str(LPD_WEIGHTS))
lpd_model.info()
print("weights:", LPD_WEIGHTS, LPD_WEIGHTS.exists())

### 1.6 Training (resumes automatically)

Each run lives in `RUNS_DIR/lpd/<name>`. If `weights/last.pt` exists from an interrupted session, training **resumes** from it; if that run already finished, training is skipped. `save_period` writes extra `epochN.pt` checkpoints during training.

In [ ]:
# TODO (done): Training
LPD_RUN_NAME = f"{Path(LPD_CFG['model']).stem}_{RUN_TAG}"
LPD_PROJECT = RUNS_DIR / "lpd"
LPD_RUN_DIR = LPD_PROJECT / LPD_RUN_NAME


def lpd_run_state(run_dir):
    last = run_dir / "weights" / "last.pt"
    if not last.exists():
        return "new"
    ckpt = torch.load(last, map_location="cpu", weights_only=False)
    return "finished" if ckpt.get("epoch", -1) == -1 else "resume"  # Ultralytics sets epoch=-1 when done


def lpd_losses_finite(run_dir):
    df = pd.read_csv(run_dir / "results.csv")
    df.columns = df.columns.str.strip()
    return bool(np.isfinite(df[[c for c in df.columns if "loss" in c]].to_numpy(dtype=float)).all())


def train_lpd(yolo_device):
    n_dev = len(yolo_device) if isinstance(yolo_device, list) else 1
    state = lpd_run_state(LPD_RUN_DIR)
    if state == "finished":
        print(f"{LPD_RUN_DIR} already finished -> skipping training")
    elif state == "resume":
        print(f"resuming from {LPD_RUN_DIR / 'weights' / 'last.pt'}")
        YOLO(str(LPD_RUN_DIR / "weights" / "last.pt")).train(resume=True, device=yolo_device, workers=NUM_WORKERS)
    else:
        YOLO(str(LPD_WEIGHTS)).train(
            data=str(LPD_DATA_YAML), project=str(LPD_PROJECT), name=LPD_RUN_NAME, exist_ok=True,
            device=yolo_device, workers=NUM_WORKERS, seed=SEED,
            **{**LPD_CFG["train"], "batch": LPD_CFG["train"]["batch"] * n_dev},  # same per-GPU batch on every GPU
        )


try:
    train_lpd(YOLO_TRAIN_DEVICE)
    _ok = lpd_losses_finite(LPD_RUN_DIR)
except Exception as e:
    if isinstance(YOLO_TRAIN_DEVICE, list):  # multi-GPU DDP failed -> continue on one GPU (resumes if last.pt exists)
        print(f"multi-GPU (DDP) training failed: {e!r}\n-> retrying on a single GPU")
        YOLO_TRAIN_DEVICE = 0
        train_lpd(YOLO_TRAIN_DEVICE)
        _ok = lpd_losses_finite(LPD_RUN_DIR)
    elif YOLO_TRAIN_DEVICE == "mps":  # MPS is less mature than CUDA; fall back only for the local smoke test
        print("YOLO training on MPS failed:", repr(e))
        _ok = False
    else:
        raise
if not _ok:
    if YOLO_TRAIN_DEVICE == "mps" and SMOKE_TEST:
        print("NaN/failed on MPS -> retrying the local smoke test on CPU")
        shutil.rmtree(LPD_RUN_DIR, ignore_errors=True)
        YOLO_DEVICE = YOLO_TRAIN_DEVICE = "cpu"
        train_lpd(YOLO_TRAIN_DEVICE)
    else:
        raise RuntimeError("YOLO training produced non-finite losses")
LPD_BEST = LPD_RUN_DIR / "weights" / "best.pt"
assert LPD_BEST.exists(), LPD_BEST
log("LPD training done, best weights:", LPD_BEST)

### 1.7 Save and reload the trained detector

In [ ]:
# TODO (done): Save and reload the trained model
LPD_ARTIFACT = ARTIFACT_DIR / "lpd_best.pt"
shutil.copy2(LPD_BEST, LPD_ARTIFACT)
# model 1 of 3 -> its own output folder, immediately (survives any later crash)
LPD_MODEL_DIR = MODELS_DIR / "1_lpd_yolo26n"
LPD_MODEL_DIR.mkdir(parents=True, exist_ok=True)
for _src, _dst in ((LPD_BEST, "best.pt"), (LPD_RUN_DIR / "weights" / "last.pt", "last.pt"), (LPD_RUN_DIR / "results.csv", "results.csv"),
                   (LPD_RUN_DIR / "args.yaml", "args.yaml"), (LPD_DATA_YAML, "data.yaml"), (LPD_RUN_DIR / "results.png", "results.png")):
    if Path(_src).exists():
        shutil.copy2(_src, LPD_MODEL_DIR / _dst)
(LPD_MODEL_DIR / "README.txt").write_text(
    "Model 1/3: YOLO26n licence-plate DETECTOR (Ultralytics).\n"
    "best.pt = weights with the best validation fitness, last.pt = final epoch, results.csv = per-epoch metrics.\n"
    "Use:  from ultralytics import YOLO; YOLO('best.pt').predict('car.jpg', imgsz=%d, conf=<threshold in metrics.json>)\n"
    % LPD_CFG["train"]["imgsz"], encoding="utf-8")
log("model 1/3 saved to", LPD_MODEL_DIR)
lpd_model = YOLO(str(LPD_ARTIFACT))
LPD_IMGSZ = LPD_CFG["train"]["imgsz"]
free_gpu()


def predict_in_chunks(model, paths, conf=0.25, chunk=16):
    """YOLO predict over many image paths, `chunk` at a time.

    Passing a whole list to predict() makes Ultralytics load every image and run them as ONE batch
    (938 test images at 960 px = CUDA out-of-memory), so large lists are always split here.
    Returns lightweight numpy results: [(xyxy [N,4], conf [N], (H, W)), ...]."""
    out = []
    paths = [str(p) for p in paths]
    for i in range(0, len(paths), chunk):
        for r in model.predict(paths[i: i + chunk], imgsz=LPD_IMGSZ, conf=conf, device=YOLO_DEVICE, verbose=False):
            out.append((r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.orig_shape))
    return out

_sample = sorted((LPD_YOLO_DIR / "images" / "val").iterdir())[0]
_res = lpd_model.predict(str(_sample), imgsz=LPD_IMGSZ, conf=0.25, device=YOLO_DEVICE, verbose=False)[0]
print(f"reloaded {LPD_ARTIFACT.name}; {_sample.name}: {len(_res.boxes)} boxes")
print("xyxy:", _res.boxes.xyxy.cpu().numpy().round(1), "conf:", _res.boxes.conf.cpu().numpy().round(3))

### Inspect Results

### 1.8 Predictions on validation images

Green boxes are ground truth; red boxes are predictions, labelled with their confidence.

In [ ]:
# TODO (done): Inspect prediction of validation samples
_val_imgs = sorted((LPD_YOLO_DIR / "images" / "val").iterdir())[:6]
_preds = predict_in_chunks(lpd_model, _val_imgs, conf=0.25)
fig, axes = plt.subplots(2, 3, figsize=(20, 13))
for a, p, r in zip(axes.ravel(), _val_imgs, _preds):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    t = max(2, W // 300)
    for l in (LPD_YOLO_DIR / "labels" / "val" / f"{p.stem}.txt").read_text().splitlines():
        _, cx, cy, bw, bh = map(float, l.split())
        cv2.rectangle(img, (int((cx - bw / 2) * W), int((cy - bh / 2) * H)), (int((cx + bw / 2) * W), int((cy + bh / 2) * H)), (0, 255, 0), t)
    for (x1, y1, x2, y2), c in zip(r[0].astype(int), r[1]):
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), t)
        cv2.putText(img, f"{c:.2f}", (x1, max(0, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, W / 1000, (255, 0, 0), t)
    a.imshow(img)
    a.set_title(f"{p.name}: {len(r[1])} predicted", fontsize=13)
    a.axis("off")
plt.suptitle("validation images: green = ground truth, red = prediction", fontsize=18)
show("lpd_val_predictions")

### 1.9 Training metadata in the run folder

In [ ]:
# TODO (done): During training Ultralytics models, training metadata store in .runs/
# Inspect and analyze them.
print(LPD_RUN_DIR)
for p in sorted(LPD_RUN_DIR.rglob("*")):
    if p.is_file():
        print(f"  {str(p.relative_to(LPD_RUN_DIR)):35s} {p.stat().st_size / 1e3:9.1f} kB")
lpd_results = pd.read_csv(LPD_RUN_DIR / "results.csv")
lpd_results.columns = lpd_results.columns.str.strip()  # Ultralytics pads column names
print("\nresults.csv columns:", list(lpd_results.columns))
display(lpd_results.tail())
lpd_results.to_csv(RESULTS_DIR / "lpd_results_per_epoch.csv", index=False)
_args = yaml.safe_load((LPD_RUN_DIR / "args.yaml").read_text())
print({k: _args[k] for k in ("model", "epochs", "imgsz", "batch", "optimizer", "lr0", "cos_lr", "device", "patience")})
if (LPD_RUN_DIR / "results.png").exists():
    plt.figure(figsize=(20, 10))
    plt.imshow(Image.open(LPD_RUN_DIR / "results.png"))
    plt.axis("off")
    show("lpd_ultralytics_results")

### 1.10 Loss, precision, recall and mAP curves

In [ ]:
# TODO (done): Plot loss, recall, precision, mAP50 and mAP50-95 during training
_cols = list(lpd_results.columns)
_loss_kinds = sorted({c.split("/", 1)[1] for c in _cols if "loss" in c and "/" in c})  # YOLO26: no dfl_loss
_metric_cols = {
    "precision": next((c for c in _cols if "precision" in c), None),
    "recall": next((c for c in _cols if "recall" in c), None),
    "mAP50": next((c for c in _cols if "mAP50(" in c), None),
    "mAP50-95": next((c for c in _cols if "mAP50-95" in c), None),
}
print("loss kinds:", _loss_kinds, "| metric columns:", _metric_cols)
_ep = np.asarray(lpd_results["epoch"]) if "epoch" in _cols else np.arange(1, len(lpd_results) + 1)
for k in _loss_kinds:  # one figure per loss type: train vs val
    plt.figure()
    for split, color in (("train", "tab:blue"), ("val", "tab:orange")):
        if f"{split}/{k}" in _cols:
            plt.plot(_ep, lpd_results[f"{split}/{k}"], marker="o", ms=4, color=color, label=split)
    plt.title(f"YOLO26 {k} per epoch")
    plt.xlabel("epoch")
    plt.ylabel(k)
    plt.legend()
    show(f"lpd_{k}")

for name, c in _metric_cols.items():  # one figure per validation metric
    if not c:
        continue
    plt.figure()
    plt.plot(_ep, lpd_results[c], marker="o", ms=4, color="tab:green")
    _best = int(lpd_results[c].to_numpy().argmax())
    plt.scatter([_ep[_best]], [lpd_results[c].iloc[_best]], s=120,
                color="red", zorder=3, label=f"best {lpd_results[c].iloc[_best]:.4f}")
    plt.title(f"YOLO26 validation {name} per epoch")
    plt.xlabel("epoch")
    plt.ylabel(name)
    plt.ylim(0, 1.02)
    plt.legend()
    show(f"lpd_val_{name.replace('-', '_')}")

plt.figure()  # all four together for a quick overview
for name, c in _metric_cols.items():
    if c:
        plt.plot(_ep, lpd_results[c], marker="o", ms=4, label=name)
plt.title("YOLO26 validation metrics per epoch")
plt.xlabel("epoch")
plt.ylim(0, 1.02)
plt.legend()
show("lpd_val_metrics_overview")

### 1.11 Final evaluation on the validation and held-out test splits

This step also derives the **F1-optimal confidence threshold** on the validation split. That threshold is the value recommended for Phase 3.

In [ ]:
# TODO (done): Inspct the final result of the trained model
_val_kw = dict(data=str(LPD_DATA_YAML), imgsz=LPD_IMGSZ, batch=LPD_CFG["train"]["batch"], device=YOLO_DEVICE,
               workers=NUM_WORKERS, project=str(LPD_PROJECT), exist_ok=True, plots=True, verbose=False)
lpd_val_metrics = lpd_model.val(split="val", name=f"{LPD_RUN_NAME}_val", **_val_kw)
lpd_test_metrics = lpd_model.val(split="test", name=f"{LPD_RUN_NAME}_test", **_val_kw)


def det_metrics(m):
    return {"precision": float(m.box.mp), "recall": float(m.box.mr), "mAP50": float(m.box.map50),
            "mAP50-95": float(m.box.map), "fitness": float(m.fitness), "speed_ms_per_image": {k: float(v) for k, v in m.speed.items()}}


try:  # F1-vs-confidence curve on val -> best threshold
    _f1 = np.asarray(lpd_val_metrics.box.f1_curve).mean(0)
    LPD_CONF_THRESHOLD = float(np.clip(np.asarray(lpd_val_metrics.box.px)[_f1.argmax()], 0.05, 0.9))
except Exception as e:
    print("could not read the F1 curve:", e)
    LPD_CONF_THRESHOLD = 0.25
LPD_METRICS = {
    "model": LPD_CFG["model"], "run": LPD_RUN_NAME, "smoke_test": SMOKE_TEST, "imgsz": LPD_IMGSZ,
    "epochs_trained": int(len(lpd_results)), "n_images": {k: len(v) for k, v in LPD_USE.items()},
    "val": det_metrics(lpd_val_metrics), "test": det_metrics(lpd_test_metrics),
    "recommended_conf_threshold": LPD_CONF_THRESHOLD, "lpd_has_plate_text": LPD_HAS_PLATE_TEXT,
}
(ARTIFACT_DIR / "lpd_metrics.json").write_text(json.dumps(LPD_METRICS, indent=2))
shutil.copy2(ARTIFACT_DIR / "lpd_metrics.json", LPD_MODEL_DIR / "metrics.json")
_lpd_tab = pd.DataFrame({s: {k: v for k, v in LPD_METRICS[s].items() if k != "speed_ms_per_image"} for s in ("val", "test")}).T
_lpd_tab.rename_axis("split").to_csv(RESULTS_DIR / "lpd_metrics.csv")
log("LPD metrics:", _lpd_tab.round(4).to_dict(orient="index"), "| recommended conf", round(LPD_CONF_THRESHOLD, 3))
display(pd.DataFrame({k: LPD_METRICS[k] for k in ("val", "test")}).drop("speed_ms_per_image").astype(float).round(4))
print("recommended confidence threshold (F1-optimal on val):", round(LPD_CONF_THRESHOLD, 3))

free_gpu()
for _split, _d in (("val", lpd_val_metrics.save_dir), ("test", lpd_test_metrics.save_dir)):  # Ultralytics' own plots
    for _pat in ("confusion_matrix_normalized.png", "*PR_curve.png", "*F1_curve.png"):
        for p in sorted(Path(_d).glob(_pat)):
            plt.figure(figsize=(12, 9))
            plt.imshow(Image.open(p))
            plt.title(f"{_split}: {p.stem}")
            plt.axis("off")
            show(f"lpd_{_split}_{p.stem}")

### 1.12 Test-set error analysis

Each ground-truth plate in the held-out test split is matched to its best prediction. A match needs IoU ≥ 0.5 and confidence ≥ the recommended threshold. The figures show:
* how tightly the matched boxes fit (IoU histogram);
* whether confidence separates true from false detections;
* **recall by plate size**, since small, distant plates are the hard case;
* the test images with the most errors (missed plates or false boxes).

Tables: `outputs/lpd_test_boxes.csv` (one row per ground-truth plate) and `outputs/lpd_test_images.csv` (one row per image).

In [ ]:
def box_iou(a, b):
    """IoU matrix between boxes a [N,4] and b [M,4] in xyxy pixels."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    tl = np.maximum(a[:, None, :2], b[None, :, :2])
    br = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(br - tl, 0, None).prod(-1)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


_test_imgs = sorted((LPD_YOLO_DIR / "images" / "test").iterdir())
_gt_rows, _img_rows, _fp_conf, _cache = [], [], [], {}
free_gpu()
# chunked prediction: a single predict() call on all ~938 paths would batch them all at once (CUDA OOM)
for p, (pb, pc, (H, W)) in zip(_test_imgs, predict_in_chunks(lpd_model, _test_imgs, conf=0.01)):
    _lines = (LPD_YOLO_DIR / "labels" / "test" / f"{p.stem}.txt").read_text().splitlines()
    gt = np.array([[(cx - bw / 2) * W, (cy - bh / 2) * H, (cx + bw / 2) * W, (cy + bh / 2) * H]
                   for _, cx, cy, bw, bh in (map(float, l.split()) for l in _lines)]).reshape(-1, 4)
    keep = pc >= LPD_CONF_THRESHOLD
    pb, kc = pb[keep], pc[keep]
    iou = box_iou(gt, pb)
    used = set()
    for gi in range(len(gt)):
        cand = [j for j in np.argsort(-iou[gi]) if j not in used and iou[gi, j] >= 0.5] if iou.shape[1] else []
        j = cand[0] if cand else -1
        if j >= 0:
            used.add(j)
        _gt_rows.append({"image": p.name, "gt_w_px": gt[gi, 2] - gt[gi, 0], "gt_h_px": gt[gi, 3] - gt[gi, 1],
                         "detected": j >= 0, "best_iou": float(iou[gi].max()) if iou.shape[1] else 0.0,
                         "conf": float(kc[j]) if j >= 0 else np.nan})
    fp = [j for j in range(len(kc)) if j not in used]
    _fp_conf += [float(kc[j]) for j in fp]
    _img_rows.append({"image": p.name, "n_gt": len(gt), "n_pred": len(kc), "tp": len(used), "fn": len(gt) - len(used),
                      "fp": len(fp), "max_conf": float(pc.max()) if len(pc) else 0.0})
    _cache[p.name] = (gt, pb, kc)
lpd_test_boxes, lpd_test_images = pd.DataFrame(_gt_rows), pd.DataFrame(_img_rows)
lpd_test_boxes.to_csv(RESULTS_DIR / "lpd_test_boxes.csv", index=False)
lpd_test_images.to_csv(RESULTS_DIR / "lpd_test_images.csv", index=False)
_n_det, _n_gt = int(lpd_test_boxes.detected.sum()), len(lpd_test_boxes)
_bad = lpd_test_images[(lpd_test_images.fn > 0) | (lpd_test_images.fp > 0)]
log(f"LPD test @ conf {LPD_CONF_THRESHOLD:.3f}: {_n_det}/{_n_gt} plates detected (recall {_n_det / max(_n_gt, 1):.4f}), "
    f"{len(_fp_conf)} false boxes, {len(_bad)}/{len(lpd_test_images)} images with at least one error")

plt.figure()
plt.hist(lpd_test_boxes.best_iou[lpd_test_boxes.detected], bins=25, range=(0.5, 1), color="tab:green")
plt.title("test: IoU of detected plates with ground truth")
plt.xlabel("IoU")
plt.ylabel("plates")
show("lpd_test_iou_hist")

plt.figure()
plt.hist([lpd_test_boxes.conf.dropna(), _fp_conf], bins=20, range=(0, 1), label=["true positive", "false positive"],
         color=["tab:green", "tab:red"])
plt.axvline(LPD_CONF_THRESHOLD, color="k", ls="--", label=f"threshold {LPD_CONF_THRESHOLD:.2f}")
plt.title("test: detection confidence, true vs false")
plt.xlabel("confidence")
plt.ylabel("detections")
plt.legend()
show("lpd_test_confidence_hist")

_hb = [0, 15, 25, 40, 60, 100, 1e5]
_hl = ["<15", "15-25", "25-40", "40-60", "60-100", ">100"]
_hc = pd.cut(lpd_test_boxes.gt_h_px, _hb, labels=_hl)
_rec = lpd_test_boxes.detected.groupby(_hc, observed=False).mean().reindex(_hl).fillna(0)
_cnt = _hc.value_counts().reindex(_hl).fillna(0).astype(int)
plt.figure()
plt.bar(range(len(_hl)), _rec.values, color="tab:blue")
for i, v in enumerate(_rec.values):
    plt.text(i, v, f"{v:.2f}", ha="center", va="bottom")
plt.xticks(range(len(_hl)), [f"{l} px\n(n={c})" for l, c in zip(_hl, _cnt)])
plt.title("test: recall by plate height (native resolution)")
plt.ylabel("recall")
plt.ylim(0, 1.1)
show("lpd_test_recall_by_plate_height")

_kinds = {"correct": int(((lpd_test_images.fn == 0) & (lpd_test_images.fp == 0)).sum()),
          "missed plate": int(((lpd_test_images.fn > 0) & (lpd_test_images.fp == 0)).sum()),
          "false box": int(((lpd_test_images.fn == 0) & (lpd_test_images.fp > 0)).sum()),
          "both": int(((lpd_test_images.fn > 0) & (lpd_test_images.fp > 0)).sum())}
plt.figure()
plt.bar(list(_kinds), list(_kinds.values()), color=["tab:green", "tab:orange", "tab:red", "tab:purple"])
for i, v in enumerate(_kinds.values()):
    plt.text(i, v, str(v), ha="center", va="bottom")
plt.title("test images by detection outcome")
plt.ylabel("images")
show("lpd_test_image_outcomes")
pd.DataFrame({"height_bin": _hl, "n_plates": _cnt.values, "recall": _rec.values}).to_csv(RESULTS_DIR / "lpd_recall_by_plate_height.csv", index=False)

_worst = _bad.assign(err=_bad.fn + _bad.fp).sort_values("err", ascending=False).head(6)
if len(_worst):
    fig, axes = plt.subplots(2, 3, figsize=(20, 13))
    for a in axes.ravel():
        a.axis("off")
    for a, row in zip(axes.ravel(), _worst.itertuples()):
        img = cv2.cvtColor(cv2.imread(str(LPD_YOLO_DIR / "images" / "test" / row.image)), cv2.COLOR_BGR2RGB)
        t = max(2, img.shape[1] // 300)
        gt, pb, kc = _cache[row.image]
        for x1, y1, x2, y2 in gt.astype(int):
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), t)
        for (x1, y1, x2, y2), c in zip(pb.astype(int), kc):
            cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), t)
            cv2.putText(img, f"{c:.2f}", (x1, max(0, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, img.shape[1] / 1200, (255, 0, 0), t)
        a.imshow(img)
        a.set_title(f"{row.image}: missed {row.fn}, false {row.fp}", fontsize=13)
    plt.suptitle("hardest test images (green = ground truth, red = prediction)", fontsize=18)
    show("lpd_test_worst_images")
else:
    print("no detection errors on the test split")
free_gpu()

## 2. License Plate Recognition - CNN

### Data Load and Prepration

In [ ]:
# CHANGED: skipped when the data is already there (local ./LPR, Kaggle input, or an earlier run);
#          -O saves the zip to DATA_ROOT instead of the current directory.
if need_download("LPR"):
    !gdown --fuzzy https://drive.google.com/file/d/1Me3Bk1mmITaf5QQE5Kvk1E1d8xaqsnhf/view?usp=drive_link -O "{LPR_ZIP}"
else:
    print("LPR download skipped:", DATA_SOURCE["LPR"])

In [ ]:
# CHANGED: same as the LPD unzip cell: skipped when present, idempotent, no hard-coded /content paths.
if need_unzip("LPR"):
    !mkdir -p "{LPR_EXTRACT_DIR}"
    !unzip -q -o "{LPR_ZIP}" -d "{LPR_EXTRACT_DIR}"
    finalize_extraction("LPR")
else:
    print("LPR unzip skipped:", DATA_SOURCE["LPR"])

### 2.1 Inspect the LPR data

What the inspection found (the cell below re-checks every point):

* `valid_samples.csv` has two columns, `image_path` and `label`. `image_path` is the file name inside `detections/`, and every row maps 1:1 to a JPG.
* Labels are stored **in the CSV** as Persian plate strings in the order `2 digits, letter, 3 digits, 2-digit region code`, e.g. `98ع89511`.
* Two label spellings need normalizing: the letter `ه` is often followed by a zero-width joiner (U+200D, a rendering artefact), and the letter *alef* is spelled out as `الف` (3 code points). After normalization every label is exactly **7 digits + 1 letter, with the letter at position 2**.
* Images are RGB crops of varying size (from about 65 to 1700 px wide), with a median aspect ratio (w/h) of about 3.5.

In [ ]:
free_gpu()
lpr_raw = pd.read_csv(LPR_DIR / "valid_samples.csv", dtype=str, encoding="utf-8", keep_default_na=False)
print(lpr_raw.shape, list(lpr_raw.columns))
display(lpr_raw.head())
_files = {p.name for p in (LPR_DIR / "detections").iterdir() if p.suffix.lower() in IMG_EXTS}
_rows = set(lpr_raw.image_path)
print(f"rows: {len(lpr_raw)}  files: {len(_files)}  rows without file: {len(_rows - _files)}  "
      f"files without row: {len(_files - _rows)}  duplicate rows: {lpr_raw.image_path.duplicated().sum()}")
print("raw label lengths:", dict(Counter(lpr_raw.label.str.len())))
_odd = Counter(ch for l in lpr_raw.label for ch in l if not ch.isdigit() and ch not in "".join(LETTER_LATIN))
print("characters that are neither digits nor letters:", {f"U+{ord(c):04X}": n for c, n in _odd.items()})
for L in sorted(set(lpr_raw.label.str.len()) - {8}):
    print(f"  e.g. length {L}:", [repr(x) for x in lpr_raw.label[lpr_raw.label.str.len() == L].head(3)])

_df = lpr_raw.rename(columns={"image_path": "filename", "label": "raw_label"})
_df["text"] = _df.raw_label.map(normalize_plate)
_parsed = _df.text.map(parse_plate)
_drop = {"missing_file": ~_df.filename.isin(_files), "duplicate_row": _df.filename.duplicated(), "bad_format": _parsed.isna()}
for k, m in _drop.items():
    print(f"dropping {int(m.sum())} rows: {k}", _df.raw_label[m].head(3).map(repr).tolist() if m.any() else "")
_keep = ~(_drop["missing_file"] | _drop["duplicate_row"] | _drop["bad_format"])
LPR_DF = _df[_keep].reset_index(drop=True)
LPR_DF["digits"] = [p[0] for p in _parsed[_keep]]
LPR_DF["letter"] = [p[1] for p in _parsed[_keep]]
print(f"kept {len(LPR_DF)} / {len(_df)} samples; normalized labels changed for "
      f"{int((_df.text != _df.raw_label).sum())} rows (ZWJ removed)")

# vocabularies built from the data (letters in Persian alphabet order)
DIGIT_VOCAB = list("0123456789")
LETTER_VOCAB = [c for c in LETTER_LATIN if c in set(LPR_DF.letter)]
LETTER_TO_IDX = {c: i for i, c in enumerate(LETTER_VOCAB)}
LETTER_VOCAB_LATIN = [LETTER_LATIN[c] for c in LETTER_VOCAB]
PLATE_LEN, LETTER_POS, DIGIT_POS = 8, 2, [0, 1, 3, 4, 5, 6, 7]
_lc = LPR_DF.letter.value_counts()
display(pd.DataFrame({"char": LETTER_VOCAB, "latin": LETTER_VOCAB_LATIN, "count": [int(_lc[c]) for c in LETTER_VOCAB]}).rename_axis("index").T)

_sizes = []
for f in tqdm(LPR_DF.filename, desc="read LPR image headers", mininterval=10):
    with Image.open(LPR_DIR / "detections" / f) as im:
        _sizes.append((*im.size, im.mode))
LPR_DF["w"], LPR_DF["h"], LPR_DF["mode"] = zip(*_sizes)
LPR_DF["aspect"] = LPR_DF.w / LPR_DF.h
print("colour modes:", dict(Counter(LPR_DF["mode"])))
display(LPR_DF[["w", "h", "aspect"]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).round(2))

### 2.2 LPR exploratory data analysis

In [ ]:
_rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(4, 3, figsize=(18, 10))
for a, i in zip(axes.ravel(), _rng.choice(len(LPR_DF), 12, replace=False)):
    r = LPR_DF.iloc[i]
    a.imshow(Image.open(LPR_DIR / "detections" / r.filename))
    a.set_title(f"{plate_to_latin(r.text)}  ({r.w}x{r.h})", fontsize=14)
    a.axis("off")
plt.suptitle("random LPR samples (label transliterated: DD-letter-DDD-region)", fontsize=18)
show("lpr_samples")

plt.figure()
plt.hist(LPR_DF.w, bins=np.logspace(np.log10(LPR_DF.w.min()), np.log10(LPR_DF.w.max()), 60), alpha=0.7, label="width")
plt.hist(LPR_DF.h, bins=np.logspace(np.log10(LPR_DF.h.min()), np.log10(LPR_DF.h.max()), 60), alpha=0.7, label="height")
plt.xscale("log")
plt.title("LPR crop sizes")
plt.xlabel("pixels (log scale)")
plt.ylabel("crops")
plt.legend()
show("lpr_crop_sizes")

plt.figure()
plt.hist(LPR_DF.aspect, bins=80)
plt.axvline(LPR_DF.aspect.median(), color="k", ls=":", label=f"median {LPR_DF.aspect.median():.2f}")
plt.axvline(4.0, color="r", ls="--", label="model input 256/64 = 4")
plt.title("LPR crop aspect ratio (width / height)")
plt.xlabel("w / h")
plt.ylabel("crops")
plt.legend()
show("lpr_crop_aspect_ratio")

plt.figure(figsize=(14, 6.5))
plt.bar(LETTER_VOCAB_LATIN, [_lc[c] for c in LETTER_VOCAB])
for i, c in enumerate(LETTER_VOCAB):
    plt.text(i, _lc[c], str(_lc[c]), ha="center", va="bottom", fontsize=10)
plt.yscale("log")
plt.xticks(rotation=60)
plt.title("letter class frequency (all LPR samples)")
plt.ylabel("samples (log scale)")
show("lpr_letter_frequency")

_pos_freq = np.zeros((7, 10), int)
for d in LPR_DF.digits:
    for j, ch in enumerate(d):
        _pos_freq[j, int(ch)] += 1
plt.figure(figsize=(12, 6.5))
plt.imshow(_pos_freq, cmap="Blues", aspect="auto")
plt.colorbar(label="count")
plt.grid(False)
plt.yticks(range(7), [f"slot {p}" for p in DIGIT_POS])
plt.xticks(range(10), DIGIT_VOCAB)
plt.xlabel("digit")
plt.title("digit frequency per label slot (slot 2 is the letter)")
show("lpr_digit_position_freq")
print("digit 0 is nearly absent in slots 0-5 (Iranian plates never start with 0); the region code (slots 6-7) is skewed.")

### 2.3 Train/validation/test split

* **Grouped by plate string.** 20,906 unique plates across 24,561 crops: the same car never appears in two splits.
* **Stratified on the letter.** Letters with fewer than 10 plates are pooled as "rare" for stratification.
* **Phase 3 leakage guard.** Most LPD plates also occur in LPR (the LPR crops come from the same photos). Every LPR plate that appears in an **LPD test image** is forced into the LPR test split. That way the recognizer used in Phase 3 has never seen the LPD test plates.

In [ ]:
# TODO (done): Train Test Split
from sklearn.model_selection import train_test_split

_lpd_test_stems = {Path(l).stem for l in (ARTIFACT_DIR / "lpd_test_split.txt").read_text().split()}
_lpd_plates = pd.read_csv(LPD_DIR / "plate_labels.csv", dtype=str, keep_default_na=False) if (LPD_DIR / "plate_labels.csv").is_file() else None
LPD_TEST_PLATES = set() if _lpd_plates is None else {
    normalize_plate(t) for r in _lpd_plates.itertuples() if Path(r.image_path).stem in _lpd_test_stems for t in r.label.split()
}

_g = LPR_DF.groupby("text")["letter"].first()
_forced = _g.index.isin(LPD_TEST_PLATES)
_rest = _g[~_forced]


def _strat_key(s, min_count):
    c = s.map(s.value_counts())
    k = s.where(c >= min_count, "rare")
    return k if k.value_counts().min() >= 2 else None  # sklearn needs >= 2 per class


_g_train, _g_hold = train_test_split(_rest.index, test_size=CFG["lpr"]["split"]["holdout"], random_state=SEED,
                                     stratify=_strat_key(_rest, 10))
_hold = _rest[_g_hold]
_g_val, _g_test = train_test_split(_hold.index, test_size=0.5, random_state=SEED, stratify=_strat_key(_hold, 2))
_split_of = {**{g: "train" for g in _g_train}, **{g: "val" for g in _g_val}, **{g: "test" for g in _g_test},
             **{g: "test" for g in _g.index[_forced]}}
LPR_DF["split"] = LPR_DF.text.map(_split_of)
assert LPR_DF.split.notna().all()
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    assert not set(LPR_DF.text[LPR_DF.split == a]) & set(LPR_DF.text[LPR_DF.split == b])
print(f"LPD-test plates found in LPR (forced to test): {int(_forced.sum())} plates, "
      f"{int(LPR_DF.text.isin(LPD_TEST_PLATES).sum())} crops")
log("LPR split:", LPR_DF.split.value_counts().to_dict(), (LPR_DF.split.value_counts(normalize=True).round(3)).to_dict())
display(pd.crosstab(LPR_DF.letter.map(LETTER_LATIN), LPR_DF.split).T)

for s in ("train", "val", "test"):
    LPR_DF.loc[LPR_DF.split == s, ["filename", "text"]].rename(columns={"text": "label"}).to_csv(
        ARTIFACT_DIR / f"lpr_{s}_split.csv", index=False, encoding="utf-8")
LPR_SPLIT_DFS = {s: LPR_DF[LPR_DF.split == s].reset_index(drop=True) for s in ("train", "val", "test")}
if SMOKE_TEST:
    LPR_SPLIT_DFS = {s: d.sample(n=min(len(d), CFG["lpr"]["smoke_n"][s]), random_state=SEED).reset_index(drop=True)
                     for s, d in LPR_SPLIT_DFS.items()}
print("used for training/eval:", {s: len(d) for s, d in LPR_SPLIT_DFS.items()})

### 2.4 Label encoding, preprocessing and augmentation

* **Encoding.** The target is a length-8 `LongTensor` in label order. Slot 2 (the letter) indexes `LETTER_VOCAB`; every other slot indexes the shared digit vocabulary `0-9`.
* **Input size.** Crops are **letterboxed to 64×256** (H×W, aspect 4, close to the median plate aspect of about 3.5). They are padded with gray (114) and never stretched.
* **Normalization.** Both models use ImageNet mean/std. The pretrained model needs it, and one shared preprocessing function keeps the Phase-3 bundle simple. The input is RGB (the data is RGB; grayscale inputs are converted to 3 channels).
* **Train-only augmentation.**
  * A random crop/pad of up to ±8% per side, imitating loose or tight YOLO boxes.
  * A small affine transform: ±4° rotation, shear, and a scale of 0.9–1.05.
  * Brightness, contrast and saturation jitter, plus blur and occasional grayscale.
  * **No horizontal flips**, because they would mirror the characters.

Preprocessing, both models and `decode()` are written to **`lpr_models.py`** by `%%writefile` cells, and the notebook imports that file. Training, evaluation and the Phase-3 bundle therefore run the *same* code.

In [ ]:
# TODO (done): Prepare data and create pytorch dataset class for this dataset.
import torchvision.transforms as T


def encode_plate(text):
    digits, letter = parse_plate(text)
    idx = [DIGIT_VOCAB.index(c) for c in digits]
    return idx[:2] + [LETTER_TO_IDX[letter]] + idx[2:]


def decode_target(target):
    t = [int(v) for v in target]
    return "".join(DIGIT_VOCAB[i] for i in t[:2]) + LETTER_VOCAB[t[2]] + "".join(DIGIT_VOCAB[i] for i in t[3:])


assert all(decode_target(encode_plate(t)) == t for t in LPR_DF.text), "encoding must round-trip"
print("encoding round-trips for all", len(LPR_DF), "labels; example:", LPR_DF.text[0], "->", encode_plate(LPR_DF.text[0]))

LPR_PARTS_DIR = LPR_MODULE_DIR / "parts"
LPR_MODULE_PATH = LPR_MODULE_DIR / "lpr_models.py"


def build_lpr_module(upto):
    """Concatenate the %%writefile parts (<= upto) into lpr_models.py and (re)import it."""
    parts = sorted(p for p in LPR_PARTS_DIR.glob("*.py") if p.name <= upto)
    LPR_MODULE_PATH.write_text("\n".join(p.read_text() for p in parts), encoding="utf-8")
    if str(LPR_MODULE_DIR) not in sys.path:
        sys.path.insert(0, str(LPR_MODULE_DIR))
    import lpr_models

    return importlib.reload(lpr_models)

In [ ]:
%%writefile "{LPR_PARTS_DIR}/00_core.py"
"""lpr_models.py: standalone License Plate Recognition code for Phase 2 (training) and Phase 3 (pipeline).

Depends only on torch, torchvision, numpy and PIL. Generated by License_Plate_Detection_Recognition_cloud.ipynb.
Typical use:
    x = preprocess(crop_rgb)                       # [3, 64, 256] float tensor
    digit_logits, letter_logits = model(x[None])   # [1, 7, 10], [1, n_letters]
    text = decode((digit_logits, letter_logits), letter_vocab)
"""
import json
import os

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models

INPUT_SIZE = (64, 256)  # (H, W): crops are letterboxed (aspect kept), never stretched
PAD_VALUE = 114  # gray padding for letterbox and augmentation fill
MEAN = (0.485, 0.456, 0.406)  # ImageNet statistics, used by both models
STD = (0.229, 0.224, 0.225)
DIGIT_VOCAB = list("0123456789")
PLATE_LEN = 8
LETTER_POS = 2  # label layout: D D L D D D D D  (2 digits, letter, 3 digits, 2-digit region)
DIGIT_POS = (0, 1, 3, 4, 5, 6, 7)

_DIGIT_TRANS = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")


def normalize_label(text):
    """Canonical plate string: ASCII digits, no ZWJ/ZWNJ/spaces, Persian (not Arabic) yeh/kaf."""
    t = str(text).strip().translate(_DIGIT_TRANS)
    for ch in ("‍", "‌", " ", "‏", "‎"):
        t = t.replace(ch, "")
    return t.replace("ي", "ی").replace("ك", "ک")


def to_pil(img, bgr=False):
    """PIL image or HxWxC uint8 numpy array (RGB, or BGR from cv2 with bgr=True) -> RGB PIL image."""
    if isinstance(img, Image.Image):
        return img.convert("RGB")
    a = np.asarray(img)
    if a.ndim == 2:
        a = np.stack([a] * 3, axis=-1)
    elif bgr:
        a = a[..., ::-1]
    return Image.fromarray(np.ascontiguousarray(a[..., :3]).astype(np.uint8)).convert("RGB")


def letterbox(img, size=INPUT_SIZE, pad_value=PAD_VALUE):
    """Resize keeping the aspect ratio so the image fits in size=(H, W), then pad (centred) with gray."""
    H, W = size
    w, h = img.size
    s = min(W / w, H / h)
    nw, nh = max(1, round(w * s)), max(1, round(h * s))
    canvas = Image.new("RGB", (W, H), (pad_value,) * 3)
    canvas.paste(img.resize((nw, nh), Image.BILINEAR), ((W - nw) // 2, (H - nh) // 2))
    return canvas


def to_tensor(img):
    """Letterboxed RGB PIL image -> normalized float tensor [3, H, W]."""
    t = torch.from_numpy(np.asarray(img, dtype=np.float32) / 255.0).permute(2, 0, 1)
    return (t - torch.tensor(MEAN).view(3, 1, 1)) / torch.tensor(STD).view(3, 1, 1)


def preprocess(img, bgr=False):
    """Eval-time preprocessing (identical to training without augmentation): letterbox + normalize."""
    return to_tensor(letterbox(to_pil(img, bgr=bgr)))


def crop_with_padding(img, xyxy, pad=0.05):
    """Crop box (x1, y1, x2, y2 in pixels) enlarged by `pad` x box size on each side, clamped to the image.
    Works for numpy arrays (H, W, C) and PIL images; returns the same type."""
    x1, y1, x2, y2 = [float(v) for v in xyxy]
    bw, bh = x2 - x1, y2 - y1
    W, H = img.size if isinstance(img, Image.Image) else (img.shape[1], img.shape[0])
    x1, y1 = max(0, int(round(x1 - pad * bw))), max(0, int(round(y1 - pad * bh)))
    x2, y2 = min(W, int(round(x2 + pad * bw))), min(H, int(round(y2 + pad * bh)))
    return img.crop((x1, y1, x2, y2)) if isinstance(img, Image.Image) else img[y1:y2, x1:x2]


def load_config(path=None):
    path = path or os.path.join(os.path.dirname(os.path.abspath(__file__)), "lpr_config.json")
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def decode_batch(logits, letter_vocab=None):
    """(digit_logits [B,7,10], letter_logits [B,L]) -> list of plate strings like '12ب34567'."""
    digit_logits, letter_logits = logits
    if digit_logits.dim() == 2:
        digit_logits, letter_logits = digit_logits[None], letter_logits[None]
    if letter_vocab is None:
        letter_vocab = load_config()["letter_vocab"]
    out = []
    for d, l in zip(digit_logits.argmax(-1).tolist(), letter_logits.argmax(-1).tolist()):
        ds = "".join(DIGIT_VOCAB[i] for i in d)
        out.append(ds[:2] + letter_vocab[l] + ds[2:])
    return out


def decode(logits, letter_vocab=None):
    """Logits of ONE plate (batch of 1 or unbatched) -> plate string. Use decode_batch for batches."""
    texts = decode_batch(logits, letter_vocab)
    if len(texts) != 1:
        raise ValueError(f"decode() expects one plate, got {len(texts)}; use decode_batch()")
    return texts[0]

In [ ]:
lpr_models = build_lpr_module(upto="00_core.py")
assert all(lpr_models.normalize_label(t) == normalize_plate(t) for t in lpr_raw.label), "module/notebook normalization differ"
LPR_INPUT_HW = lpr_models.INPUT_SIZE
assert (LETTER_POS, tuple(DIGIT_POS), PLATE_LEN) == (lpr_models.LETTER_POS, lpr_models.DIGIT_POS, lpr_models.PLATE_LEN)


def random_crop_pad(img, max_frac):
    """Move each side of the crop in/out by up to max_frac of the size (edge-replicate when growing)."""
    a = np.asarray(img)
    h, w = a.shape[:2]
    l, t, r, b = (int(round(random.uniform(-max_frac, max_frac) * s)) for s in (w, h, w, h))
    a = np.pad(a, ((max(t, 0), max(b, 0)), (max(l, 0), max(r, 0)), (0, 0)), mode="edge")
    H2, W2 = a.shape[:2]
    a = a[max(-t, 0): H2 - max(-b, 0), max(-l, 0): W2 - max(-r, 0)]
    return Image.fromarray(a) if a.shape[0] >= 4 and a.shape[1] >= 4 else img


class TrainAugment:
    """Train-only: crop jitter -> letterbox -> small affine + photometric noise (no horizontal flip)."""

    def __init__(self, jitter):
        self.jitter = jitter
        self.tf = T.Compose([
            T.RandomApply([T.RandomAffine(degrees=4, translate=(0.03, 0.05), scale=(0.9, 1.05),
                                          shear=(-6, 6, -2, 2), fill=lpr_models.PAD_VALUE)], p=0.7),
            T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.03),
            T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.3),
            T.RandomGrayscale(p=0.1),
        ])

    def __call__(self, img):
        return self.tf(lpr_models.letterbox(random_crop_pad(img, self.jitter)))


_demo = LPR_DF.iloc[0]
_img = Image.open(LPR_DIR / "detections" / _demo.filename).convert("RGB")
_aug = TrainAugment(CFG["lpr"]["crop_jitter"])
fig, axes = plt.subplots(3, 2, figsize=(16, 9))
axes = axes.ravel()
axes[0].imshow(_img)
axes[0].set_title(f"original crop {_img.size[0]}x{_img.size[1]}", fontsize=14)
axes[1].imshow(lpr_models.letterbox(_img))
axes[1].set_title("eval input: letterbox 64x256", fontsize=14)
for a in axes[2:]:
    a.imshow(_aug(_img))
    a.set_title("random train augmentation", fontsize=14)
for a in axes:
    a.axis("off")
plt.suptitle(f"preprocessing and augmentation of {plate_to_latin(_demo.text)}", fontsize=18)
show("lpr_augmentation_demo")

The `PLPRDataset` skeleton below keeps its name and methods. `__init__` gains the arguments it needs, all with defaults.

In [ ]:
class PLPRDataset(data.Dataset):
    """Cropped plates -> (image tensor [3, 64, 256], target LongTensor[8]).

    target[LETTER_POS] indexes LETTER_VOCAB, every other slot indexes DIGIT_VOCAB (0-9).
    """

    def __init__(self, df=None, img_dir=None, augment=None):
        self.files = [] if df is None else df.filename.tolist()
        self.labels = [] if df is None else df.text.tolist()
        self.img_dir = Path(img_dir) if img_dir is not None else LPR_DIR / "detections"
        self.augment = augment  # TrainAugment for the train split, None -> eval preprocessing
        self.targets = [torch.tensor(encode_plate(t), dtype=torch.long) for t in self.labels]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        with Image.open(self.img_dir / self.files[idx]) as im:
            img = im.convert("RGB")
        img = self.augment(img) if self.augment is not None else lpr_models.letterbox(img)
        return lpr_models.to_tensor(img), self.targets[idx]

### 2.5 DataLoaders

This step also checks that one augmented training batch decodes back to its labels, which proves the encoding round-trips.

In [ ]:
# TODO (done): Define batch size and create dataloader object.
BATCH_SIZE = CFG["lpr"]["batch_size"]
lpr_datasets = {
    s: PLPRDataset(d, LPR_DIR / "detections", augment=TrainAugment(CFG["lpr"]["crop_jitter"]) if s == "train" else None)
    for s, d in LPR_SPLIT_DFS.items()
}


def make_loader(ds, train):
    return data.DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=train, drop_last=train,  # drop_last: no size-1 BatchNorm batch
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=NUM_WORKERS > 0,
        generator=torch.Generator().manual_seed(SEED) if train else None,
    )


lpr_loaders = {s: make_loader(ds, s == "train") for s, ds in lpr_datasets.items()}
print({s: f"{len(ds)} samples / {len(lpr_loaders[s])} batches" for s, ds in lpr_datasets.items()})

for i in range(8):  # dataset targets decode back to the CSV labels
    assert decode_target(lpr_datasets["train"][i][1]) == lpr_datasets["train"].labels[i]
_xb, _yb = next(iter(lpr_loaders["train"]))
print("batch:", tuple(_xb.shape), _xb.dtype, tuple(_yb.shape), _yb.dtype, f"x range [{_xb.min():.2f}, {_xb.max():.2f}]")
_mean, _std = torch.tensor(lpr_models.MEAN).view(3, 1, 1), torch.tensor(lpr_models.STD).view(3, 1, 1)
fig, axes = plt.subplots(6, 2, figsize=(16, 14))
for a, x, y in zip(axes.ravel(), _xb, _yb):
    a.imshow((x * _std + _mean).clamp(0, 1).permute(1, 2, 0).numpy())
    a.set_title(plate_to_latin(decode_target(y)), fontsize=14)
    a.axis("off")
plt.suptitle("augmented training batch with decoded targets", fontsize=18)
show("lpr_train_batch")

### Model Definition & Train

#### I. Design your model and train from scratch

**Architecture (Model I, from scratch).**

The model is a ResNet-style CNN written to `lpr_models.py`:

* **Stem.** A 3×3 conv (32 channels) with BatchNorm and ReLU.
* **Four residual stages** of 64, 128, 256 and 256 channels. Each block is conv-BN-ReLU-conv-BN plus a skip connection (a 1×1 projection when the shape changes). Stages 1–3 halve both H and W. Stage 4 halves only H (stride (2,1)), so the 64×256 input ends as a **4×32** feature map that keeps the left-to-right layout of the characters. Light `Dropout2d` in stages 3–4.
* **Head (`LPRHead`, shared with Model II):**
  * 1×1 conv reduction, BN and ReLU.
  * Adaptive average pooling to **1×16 columns**, then flatten. The flattened vector still carries *where* along the plate each feature came from.
  * FC to 512, **LayerNorm**, ReLU and **Dropout**.
  * **Multi-head output:** 7 independent digit classifiers (one 10-way head per digit slot, implemented as one `Linear(512, 7·10)`) and 1 letter classifier over `LETTER_VOCAB`.

  Independent per-slot heads were chosen over a single shared digit head because the flattened features are not aligned per character. Each slot can then learn to read its own region of the plate, and the letter gets its own vocabulary and loss weighting.
* **Initialization.** Kaiming-normal for every conv and hidden linear layer (fan-out, ReLU), BN γ=1 and β=0, the last BN of each residual branch set to γ=0 (every block starts as the identity, which helps a stable start), and small normal(0, 0.01) init for the output heads.
* **Loss.** The mean of the 8 per-slot cross-entropies with label smoothing 0.05. The letter CE uses class weights ∝ (1/freq)^0.5, capped at 5, because the letter classes are very imbalanced (`د` ≈ 2.3k samples versus `ز`, `پ` with 1 each).

In [ ]:
%%writefile "{LPR_PARTS_DIR}/10_scratch_model.py"


class LPRHead(nn.Module):
    """Multi-head output shared by both models: 7 independent digit classifiers + 1 letter classifier."""

    def __init__(self, in_ch, n_letters, mid_ch=256, pool_w=8, hidden=512, dropout=0.3):
        super().__init__()
        self.n_digit_slots = len(DIGIT_POS)
        self.reduce = nn.Sequential(nn.Conv2d(in_ch, mid_ch, 1, bias=False), nn.BatchNorm2d(mid_ch), nn.ReLU(inplace=True))
        self.pool = nn.AdaptiveAvgPool2d((1, pool_w))  # pool height, keep pool_w columns (left -> right)
        self.fc = nn.Linear(mid_ch * pool_w, hidden)
        self.norm = nn.LayerNorm(hidden)
        self.act = nn.ReLU(inplace=True)
        self.drop = nn.Dropout(dropout)
        self.digit_head = nn.Linear(hidden, self.n_digit_slots * len(DIGIT_VOCAB))  # = 7 separate 10-way heads
        self.letter_head = nn.Linear(hidden, n_letters)
        nn.init.kaiming_normal_(self.reduce[0].weight, mode="fan_out", nonlinearity="relu")
        nn.init.kaiming_normal_(self.fc.weight, nonlinearity="relu")
        nn.init.zeros_(self.fc.bias)
        for head in (self.digit_head, self.letter_head):
            nn.init.normal_(head.weight, std=0.01)
            nn.init.zeros_(head.bias)

    def forward(self, f):
        x = self.drop(self.pool(self.reduce(f)).flatten(1))
        x = self.drop(self.act(self.norm(self.fc(x))))
        return self.digit_head(x).view(-1, self.n_digit_slots, len(DIGIT_VOCAB)), self.letter_head(x)


class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, drop=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.relu1 = nn.ReLU(inplace=True)
        self.drop = nn.Dropout2d(drop) if drop > 0 else nn.Identity()
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu2 = nn.ReLU(inplace=True)
        self.shortcut = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride, bias=False), nn.BatchNorm2d(out_ch))

    def forward(self, x):
        y = self.bn2(self.conv2(self.drop(self.relu1(self.bn1(self.conv1(x))))))
        return self.relu2(y + self.shortcut(x))


class ScratchLPRNet(nn.Module):
    """Model I: residual CNN trained from scratch. Input [B,3,64,256] -> ([B,7,10], [B,n_letters])."""

    def __init__(self, n_letters, widths=(64, 128, 256, 256), dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 32, 3, 1, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        strides = (2, 2, 2, (2, 1))  # 64x256 -> 32x128 -> 16x64 -> 8x32 -> 4x32
        drops = (0.0, 0.0, 0.05, 0.1)
        stages, c = [], 32
        for w, s, d in zip(widths, strides, drops):
            stages.append(nn.Sequential(ResidualBlock(c, w, s, d), ResidualBlock(w, w, 1, d)))
            c = w
        self.stages = nn.Sequential(*stages)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
        for m in self.modules():
            if isinstance(m, ResidualBlock):
                nn.init.zeros_(m.bn2.weight)  # each residual branch starts as identity
        self.head = LPRHead(c, n_letters, mid_ch=256, pool_w=16, dropout=dropout)

    def forward(self, x):
        return self.head(self.stages(self.stem(x)))

In [ ]:
lpr_models = build_lpr_module(upto="10_scratch_model.py")
N_LETTERS = len(LETTER_VOCAB)


def count_params(m):
    return sum(p.numel() for p in m.parameters()), sum(p.numel() for p in m.parameters() if p.requires_grad)


set_seed()
scratch_model = lpr_models.ScratchLPRNet(N_LETTERS)
with torch.no_grad():
    _d, _l = scratch_model.eval()(torch.zeros(2, 3, *LPR_INPUT_HW))
print("output shapes:", tuple(_d.shape), tuple(_l.shape), "| params (total, trainable):", count_params(scratch_model))
print(scratch_model)

### 2.6 Shared training loop, used by both models

* **Optimization.** AdamW with no weight decay on biases or norm parameters, and a **OneCycle** schedule (warm-up, then cosine decay, stepped per batch; one max LR per parameter group). Gradients are clipped to a global norm of 5.
* **Precision.** AMP (fp16 autocast + GradScaler) only on CUDA; fp32 on MPS and CPU.
* **Multiple GPUs.** With more than one GPU (Kaggle T4 x2), each training batch is split across the GPUs with `nn.DataParallel`. Checkpoints always store the plain, unwrapped model.
* **Early stopping** on validation **full-plate accuracy**, meaning all 8 characters correct.
* **Checkpoints.** `RUNS_DIR/lpr/<run>/last.pt` is written every epoch: model, optimizer, scheduler, scaler, history and early-stopping state. Re-running a cell **resumes** from it, and a finished run is simply reloaded. `best.pt` holds only the best `state_dict`.
* **Logged every epoch:** train/val loss, per-character and full-plate accuracy, and LR.
* **Logged every `log_every` steps:** the global gradient norm (before clipping), per-layer gradient norms for representative layers (first, middle and last backbone conv, head FC, letter head), the fraction of zero outputs of selected ReLUs (dead activations), and the fraction of letter predictions with softmax > 0.99 (output saturation).

In [ ]:
LPR_RUNS = RUNS_DIR / "lpr"
LPR_RUNS.mkdir(parents=True, exist_ok=True)
LOG_EVERY = CFG["lpr"]["log_every"]


def letter_class_weights(train_df):
    counts = train_df.letter.value_counts().reindex(LETTER_VOCAB, fill_value=0).to_numpy(float)
    w = np.where(counts > 0, (counts.sum() / np.maximum(counts, 1)) ** CFG["lpr"]["letter_weight_power"], 0.0)
    w = np.clip(w / w[counts > 0].mean(), 0.0, CFG["lpr"]["max_letter_weight"])
    return torch.tensor(w, dtype=torch.float32)


class PlateLoss(nn.Module):
    """Mean of the 8 per-slot cross-entropies (7 digit slots + 1 letter slot)."""

    def __init__(self, letter_weights=None, label_smoothing=0.0):
        super().__init__()
        self.digit_ce = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        self.letter_ce = nn.CrossEntropyLoss(weight=letter_weights, label_smoothing=label_smoothing)

    def forward(self, outputs, target):
        digit_logits, letter_logits = outputs
        d = self.digit_ce(digit_logits.float().reshape(-1, len(DIGIT_VOCAB)), target[:, DIGIT_POS].reshape(-1))
        l = self.letter_ce(letter_logits.float(), target[:, LETTER_POS])
        return (len(DIGIT_POS) * d + l) / PLATE_LEN


def predict_targets(outputs):
    digit_logits, letter_logits = outputs
    pred = torch.empty(digit_logits.shape[0], PLATE_LEN, dtype=torch.long, device=digit_logits.device)
    pred[:, DIGIT_POS] = digit_logits.argmax(-1)
    pred[:, LETTER_POS] = letter_logits.argmax(-1)
    return pred


def autocast_ctx():
    return torch.autocast("cuda", dtype=torch.float16) if USE_AMP else contextlib.nullcontext()


def sync_device(dev):
    if dev.type == "cuda":
        torch.cuda.synchronize()
    elif dev.type == "mps":
        torch.mps.synchronize()


EVAL_CRITERION = PlateLoss().to(DEVICE)  # plain CE (no smoothing / weights) -> comparable val/test loss
TRAIN_CRITERION = PlateLoss(letter_class_weights(LPR_SPLIT_DFS["train"]), CFG["lpr"]["label_smoothing"]).to(DEVICE)


@torch.no_grad()
def evaluate(model, loader, criterion=EVAL_CRITERION, return_preds=False):
    model.eval()
    tot, n, preds, targets = 0.0, 0, [], []
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with autocast_ctx():
            out = model(x)
        out = tuple(o.float() for o in out)
        tot += criterion(out, y).item() * len(y)
        n += len(y)
        preds.append(predict_targets(out).cpu())
        targets.append(y.cpu())
    P, Y = torch.cat(preds), torch.cat(targets)
    ok = P == Y
    res = {"loss": tot / n, "char_acc": ok.float().mean().item(), "plate_acc": ok.all(1).float().mean().item(),
           "pos_acc": ok.float().mean(0).tolist()}
    if return_preds:
        res["preds"], res["targets"] = P, Y
    return res


def param_groups(module, lr, weight_decay):
    decay, no_decay = [], []
    for n, p in module.named_parameters():
        if p.requires_grad:
            (no_decay if p.ndim <= 1 else decay).append(p)
    return [{"params": decay, "lr": lr, "weight_decay": weight_decay}, {"params": no_decay, "lr": lr, "weight_decay": 0.0}]


def pick_monitor_layers(model):
    convs = [n for n, p in model.named_parameters() if p.ndim == 4 and not n.startswith("head.")]
    return [convs[0], convs[len(convs) // 2], convs[-1], "head.fc.weight", "head.letter_head.weight"]


def pick_activation_modules(model):
    relus = [(n, m) for n, m in model.named_modules() if isinstance(m, nn.ReLU) and not n.startswith("head.")]
    picks = dict([relus[0], relus[len(relus) // 2], relus[-1]]) if relus else {}
    picks.update({"head.reduce.2": model.head.reduce[2], "head.act": model.head.act})
    return picks


class ActivationMonitor:
    """Forward hooks that record the fraction of zero (dead) ReLU outputs on logging steps only."""

    def __init__(self, modules):
        self.active, self.stats = False, {}
        self.handles = [m.register_forward_hook(self._hook(n)) for n, m in modules.items()]

    def _hook(self, name):
        def fn(_m, _i, out):
            if self.active:
                self.stats[name] = (out.detach() <= 0).float().mean().item()
        return fn

    def remove(self):
        for h in self.handles:
            h.remove()


def _new_history(layer_names, act_names):
    keys = ["epoch", "train_loss", "val_loss", "train_char_acc", "val_char_acc", "train_plate_acc", "val_plate_acc",
            "val_pos_acc", "lr", "epoch_time"]
    return {**{k: [] for k in keys}, "steps": {"step": [], "grad_norm": [], "lr": [], "saturated": [],
            "layers": {n: [] for n in layer_names}, "act_zero": {n: [] for n in act_names}}}


def fit(model, run_name, groups, epochs, patience, criterion=TRAIN_CRITERION, init_state=None):
    """Shared training loop with resume. Returns (history, best_checkpoint_path); model ends with best weights."""
    run_dir = LPR_RUNS / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    last_path, best_path = run_dir / "last.pt", run_dir / "best.pt"
    model.to(DEVICE)
    optimizer = optim.AdamW(groups)
    steps = len(lpr_loaders["train"])
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=[g["lr"] for g in groups], epochs=epochs,
                                              steps_per_epoch=steps, pct_start=CFG["lpr"]["pct_start"])
    scaler = torch.amp.GradScaler("cuda") if USE_AMP else None
    layer_names, act_modules = pick_monitor_layers(model), pick_activation_modules(model)
    history = _new_history(layer_names, list(act_modules))
    start_epoch, best_acc, best_epoch, bad_epochs, global_step = 0, -1.0, -1, 0, 0

    if last_path.exists():
        ck = torch.load(last_path, map_location="cpu", weights_only=False)
        history, best_acc, best_epoch, bad_epochs = ck["history"], ck["best_acc"], ck["best_epoch"], ck["bad_epochs"]
        if ck.get("done"):
            model.load_state_dict(torch.load(best_path, map_location="cpu", weights_only=True)["state_dict"])
            log(f"[{run_name}] already finished: best val plate acc {best_acc:.4f} (epoch {best_epoch + 1}) -> loaded")
            return history, best_path
        model.load_state_dict(ck["model"])
        optimizer.load_state_dict(ck["optimizer"])
        scheduler.load_state_dict(ck["scheduler"])
        if scaler is not None and ck.get("scaler"):
            scaler.load_state_dict(ck["scaler"])
        start_epoch, global_step = ck["epoch"] + 1, ck["global_step"]
        log(f"[{run_name}] resuming at epoch {start_epoch + 1}/{epochs}")
    elif init_state is not None:
        model.load_state_dict(init_state)

    params = dict(model.named_parameters())
    trainable = [p for g in groups for p in g["params"]]
    net = nn.DataParallel(model) if (N_GPUS > 1 and LPR_MULTI_GPU) else model  # optional; checkpoints stay unwrapped
    mon = ActivationMonitor(act_modules)
    try:
        for epoch in range(start_epoch, epochs):
            model.train()
            t0 = time.time()
            loss_sum = torch.zeros((), device=DEVICE)
            char_ok = torch.zeros((), device=DEVICE)
            plate_ok = torch.zeros((), device=DEVICE)
            seen = 0
            for x, y in tqdm(lpr_loaders["train"], desc=f"{run_name} {epoch + 1}/{epochs}", leave=False, mininterval=30):
                log_now = global_step % LOG_EVERY == 0
                mon.active = log_now
                x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with autocast_ctx():
                    out = net(x)
                    loss = criterion(out, y)
                if scaler is not None:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                else:
                    loss.backward()
                if log_now:
                    lnorms = {n: params[n].grad.norm().item() if params[n].grad is not None else float("nan") for n in layer_names}
                gnorm = nn.utils.clip_grad_norm_(trainable, CFG["lpr"]["grad_clip"])
                if scaler is not None:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                scheduler.step()
                with torch.no_grad():
                    ok = predict_targets(tuple(o.detach() for o in out)) == y
                    char_ok += ok.float().sum()
                    plate_ok += ok.all(1).sum()
                    loss_sum += loss.detach() * len(y)
                    seen += len(y)
                if log_now:
                    st = history["steps"]
                    st["step"].append(global_step)
                    st["grad_norm"].append(gnorm.item())
                    st["lr"].append([g["lr"] for g in optimizer.param_groups])
                    st["saturated"].append((torch.softmax(out[1].detach().float(), -1).amax(-1) > 0.99).float().mean().item())
                    for n in layer_names:
                        st["layers"][n].append(lnorms[n])
                    for n in st["act_zero"]:
                        st["act_zero"][n].append(mon.stats.get(n, float("nan")))
                global_step += 1
            mon.active = False
            val = evaluate(model, lpr_loaders["val"])
            h = history
            h["epoch"].append(epoch + 1)
            h["train_loss"].append(loss_sum.item() / seen)
            h["train_char_acc"].append(char_ok.item() / (seen * PLATE_LEN))
            h["train_plate_acc"].append(plate_ok.item() / seen)
            h["val_loss"].append(val["loss"])
            h["val_char_acc"].append(val["char_acc"])
            h["val_plate_acc"].append(val["plate_acc"])
            h["val_pos_acc"].append(val["pos_acc"])
            h["lr"].append([g["lr"] for g in optimizer.param_groups])
            h["epoch_time"].append(time.time() - t0)
            if val["plate_acc"] > best_acc:
                best_acc, best_epoch, bad_epochs = val["plate_acc"], epoch, 0
                torch.save({"state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                            "epoch": epoch, "val": {k: val[k] for k in ("loss", "char_acc", "plate_acc")}}, best_path)
            else:
                bad_epochs += 1
            done = bad_epochs >= patience or epoch == epochs - 1
            torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                        "scaler": scaler.state_dict() if scaler is not None else None, "epoch": epoch,
                        "global_step": global_step, "history": history, "best_acc": best_acc, "best_epoch": best_epoch,
                        "bad_epochs": bad_epochs, "done": done}, last_path)
            log(f"[{run_name}] ep {epoch + 1:3d}/{epochs} | loss {h['train_loss'][-1]:.4f}/{val['loss']:.4f} | "
                  f"char {h['train_char_acc'][-1]:.4f}/{val['char_acc']:.4f} | plate {h['train_plate_acc'][-1]:.4f}/"
                  f"{val['plate_acc']:.4f} | lr {optimizer.param_groups[0]['lr']:.2e} | {h['epoch_time'][-1]:.0f}s"
                  + ("  *best*" if bad_epochs == 0 else ""))
            if done:
                if bad_epochs >= patience:
                    log(f"[{run_name}] early stop: no val plate-acc improvement for {patience} epochs")
                break
    finally:
        mon.remove()
    model.load_state_dict(torch.load(best_path, map_location="cpu", weights_only=True)["state_dict"])
    log(f"[{run_name}] best val plate acc {best_acc:.4f} at epoch {best_epoch + 1}")
    return history, best_path


def concat_histories(h1, h2):
    """Join stage-1 and stage-2 histories (epochs / steps continue) for plotting."""
    out = copy.deepcopy(h1)
    e0, s0 = (h1["epoch"][-1] if h1["epoch"] else 0), (h1["steps"]["step"][-1] + 1 if h1["steps"]["step"] else 0)
    for k, v in h2.items():
        if k == "steps":
            continue
        out[k] += [e + e0 for e in v] if k == "epoch" else v
    for k in ("grad_norm", "saturated"):
        out["steps"][k] += h2["steps"][k]
    out["steps"]["step"] += [s + s0 for s in h2["steps"]["step"]]
    out["steps"]["lr"] += [l[:1] + l[-1:] for l in h2["steps"]["lr"]]
    out["steps"]["lr"] = [l[:1] + l[-1:] for l in out["steps"]["lr"]]
    for grp in ("layers", "act_zero"):
        for n, v in h2["steps"][grp].items():
            out["steps"][grp].setdefault(n, [float("nan")] * (len(out["steps"]["step"]) - len(v)))
            out["steps"][grp][n] += v
        for n in out["steps"][grp]:
            if n not in h2["steps"][grp]:
                out["steps"][grp][n] += [float("nan")] * len(h2["steps"]["step"])
    return out


LPR_MODEL_DIRS = {"scratch": MODELS_DIR / "2_lpr_scratch", "pretrained": MODELS_DIR / "3_lpr_pretrained"}


def save_lpr_model_dir(name, model, history):
    """Model 2/3 or 3/3 -> its own output folder right after training: state_dict, TorchScript, code, vocab, history."""
    d = LPR_MODEL_DIRS[name]
    d.mkdir(parents=True, exist_ok=True)
    backbone = CFG["lpr"]["pretrained"]["backbone"]
    sd = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    torch.save({"state_dict": sd, "arch": name, "n_letters": N_LETTERS, "letter_vocab": LETTER_VOCAB,
                "input_size": list(LPR_INPUT_HW), "backbone": backbone}, d / "best_state_dict.pt")
    cpu_model = (lpr_models.ScratchLPRNet(N_LETTERS) if name == "scratch"
                 else lpr_models.PretrainedLPRNet(N_LETTERS, pretrained=False, backbone=backbone))
    cpu_model.load_state_dict(sd)
    cpu_model.eval()
    with torch.no_grad():
        torch.jit.trace(cpu_model, torch.zeros(1, 3, *LPR_INPUT_HW)).save(str(d / "model.ts"))
    shutil.copy2(LPR_MODULE_PATH, d / "lpr_models.py")
    (d / "vocab.json").write_text(json.dumps({
        "arch": name, "input_size_hw": list(LPR_INPUT_HW), "color_mode": "RGB",
        "normalization": {"mean": list(lpr_models.MEAN), "std": list(lpr_models.STD)}, "digit_vocab": DIGIT_VOCAB,
        "letter_vocab": LETTER_VOCAB, "letter_vocab_latin": LETTER_VOCAB_LATIN, "letter_position": LETTER_POS,
        "output_format": "DD L DDD DD, e.g. 12ب34567"}, indent=2, ensure_ascii=False), encoding="utf-8")
    save_history_csv(history, name)
    for f in (f"lpr_{name}_history.csv", f"lpr_{name}_steps.csv"):
        shutil.copy2(RESULTS_DIR / f, d / f.replace(f"lpr_{name}_", ""))
    (d / "README.txt").write_text(
        f"Model {'2' if name == 'scratch' else '3'}/3: licence-plate RECOGNIZER ({name}).\n"
        "model.ts           TorchScript, no class code needed: torch.jit.load('model.ts')\n"
        "best_state_dict.pt weights (+ arch, vocab); load with lpr_models.py\n"
        "lpr_models.py      preprocess(), decode(), model classes\n"
        "vocab.json         letter/digit vocabularies and preprocessing settings\n"
        "history.csv        per-epoch loss/accuracy, steps.csv = gradient-health log\n"
        "Use:  import torch, json, lpr_models as M; m = torch.jit.load('model.ts')\n"
        "      out = m(M.preprocess(plate_crop_rgb)[None]); print(M.decode(out, json.load(open('vocab.json'))['letter_vocab']))\n",
        encoding="utf-8")
    log(f"model {'2' if name == 'scratch' else '3'}/3 ({name}) saved to", d)


def save_history_csv(h, name):
    """Per-epoch metrics and per-step gradient-health log of one model as CSV."""
    ep = pd.DataFrame({k: h[k] for k in ("epoch", "train_loss", "val_loss", "train_char_acc", "val_char_acc",
                                         "train_plate_acc", "val_plate_acc", "epoch_time")})
    ep["lr_group0"] = [l[0] for l in h["lr"]]
    for i in range(PLATE_LEN):
        ep[f"val_acc_slot{i}"] = [pa[i] for pa in h["val_pos_acc"]]
    ep.to_csv(RESULTS_DIR / f"lpr_{name}_history.csv", index=False)
    st = h["steps"]
    pd.DataFrame({"step": st["step"], "grad_norm": st["grad_norm"], "lr_group0": [l[0] for l in st["lr"]],
                  "letter_softmax_gt_0.99": st["saturated"], **{f"grad_{n}": v for n, v in st["layers"].items()},
                  **{f"zero_frac_{n}": v for n, v in st["act_zero"].items()}}).to_csv(RESULTS_DIR / f"lpr_{name}_steps.csv", index=False)


def plot_history(h, title, name):
    """Six separate figures per model: loss, accuracy, LR, global grad norm, per-layer grad norm, activations."""
    st = h["steps"]
    plt.figure()
    plt.plot(h["epoch"], h["train_loss"], "o-", label="train (weighted, smoothed CE)")
    plt.plot(h["epoch"], h["val_loss"], "s-", label="val (plain CE)")
    plt.title(f"{title}: loss per epoch")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.legend()
    show(f"lpr_{name}_loss")

    plt.figure()
    for k, s in (("char", "-"), ("plate", "--")):
        plt.plot(h["epoch"], h[f"train_{k}_acc"], "o" + s, label=f"train {'per-char' if k == 'char' else 'full-plate'}")
        plt.plot(h["epoch"], h[f"val_{k}_acc"], "s" + s, label=f"val {'per-char' if k == 'char' else 'full-plate'}")
    plt.title(f"{title}: accuracy per epoch")
    plt.xlabel("epoch")
    plt.ylabel("accuracy")
    plt.ylim(0, 1.02)
    plt.legend()
    show(f"lpr_{name}_accuracy")

    plt.figure()
    lrs = np.array(st["lr"], dtype=float)
    for j in range(lrs.shape[1] if lrs.ndim == 2 else 0):
        plt.plot(st["step"], lrs[:, j], label=["backbone / first group", "head / last group"][min(j, 1)] if lrs.shape[1] > 1 else "lr")
    plt.title(f"{title}: learning-rate schedule")
    plt.xlabel("training step")
    plt.ylabel("learning rate")
    plt.legend()
    show(f"lpr_{name}_lr_schedule")

    plt.figure()
    plt.plot(st["step"], st["grad_norm"], lw=1)
    plt.axhline(CFG["lpr"]["grad_clip"], color="r", ls="--", label="clip threshold")
    plt.yscale("log")
    plt.title(f"{title}: global gradient norm (before clipping)")
    plt.xlabel("training step")
    plt.ylabel("L2 norm (log scale)")
    plt.legend()
    show(f"lpr_{name}_grad_norm")

    plt.figure()
    for n, v in st["layers"].items():
        plt.plot(st["step"], v, lw=1, label=n)
    plt.yscale("log")
    plt.title(f"{title}: per-layer gradient norm")
    plt.xlabel("training step")
    plt.ylabel("L2 norm (log scale)")
    plt.legend(fontsize=10)
    show(f"lpr_{name}_layer_grad_norms")

    plt.figure()
    for n, v in st["act_zero"].items():
        plt.plot(st["step"], v, lw=1, label=f"zero fraction {n}")
    plt.plot(st["step"], st["saturated"], "k--", lw=1.2, label="letter softmax > 0.99")
    plt.title(f"{title}: dead activations and output saturation")
    plt.xlabel("training step")
    plt.ylabel("fraction")
    plt.ylim(-0.02, 1.02)
    plt.legend(fontsize=10)
    show(f"lpr_{name}_activations")


def grad_health_report(h, name):
    st = h["steps"]
    g = np.array(st["grad_norm"], dtype=float)
    fin = np.isfinite(g)
    log(f"--- gradient health: {name} ({len(g)} logged steps)")
    if not fin.any():
        log("  no finite gradient norms logged")
        return
    log(f"  global norm: median {np.median(g[fin]):.3g}, p99 {np.percentile(g[fin], 99):.3g}, max {g[fin].max():.3g}, "
          f"min {g[fin].min():.3g}; non-finite {int((~fin).sum())} (AMP overflow steps are skipped by GradScaler); "
          f"clipped {np.mean(g[fin] > CFG['lpr']['grad_clip']):.1%}")
    med = {n: np.nanmedian(np.array(v, dtype=float)) if np.isfinite(np.array(v, dtype=float)).any() else float("nan")
           for n, v in st["layers"].items()}
    log("  per-layer median grad norm:", {n: f"{v:.2e}" for n, v in med.items()})
    finite_med = [v for v in med.values() if np.isfinite(v) and v > 0]
    if len(finite_med) >= 2:
        ratio = min(finite_med) / max(finite_med)
        log(f"  smallest/largest layer ratio {ratio:.1e} -> {'OK' if ratio > 1e-4 else 'possible VANISHING gradient'}")
    dead = {n: np.nanmean(np.array(v, dtype=float)) for n, v in st["act_zero"].items() if np.isfinite(np.array(v, dtype=float)).any()}
    log("  mean zero-activation fraction:", {n: f"{v:.2f}" for n, v in dead.items()},
          "-> " + ("OK" if all(v < 0.95 for v in dead.values()) else "some layer is (almost) DEAD"))
    sat = np.nanmean(np.array(st["saturated"][-max(1, len(st["saturated"]) // 5):], dtype=float))
    log(f"  letter predictions with softmax > 0.99 (last 20% of steps): {sat:.1%}")
    log("  verdict:", "no exploding gradients" if fin.all() and g[fin].max() < 1e3 else "check for EXPLODING gradients")

### 2.7 Train Model I (from scratch)

In [ ]:
set_seed()
_sc = CFG["lpr"]["scratch"]
scratch_model = lpr_models.ScratchLPRNet(N_LETTERS)
scratch_history, scratch_best_path = fit(
    scratch_model, f"scratch_{RUN_TAG}", param_groups(scratch_model, _sc["lr"], _sc["weight_decay"]),
    epochs=_sc["epochs"], patience=_sc["patience"],
)

In [ ]:
save_lpr_model_dir("scratch", scratch_model, scratch_history)  # saved first, before any plotting
plot_history(scratch_history, "Model I: scratch CNN", "scratch")
grad_health_report(scratch_history, "scratch")

#### II. Use pre-trained models as backbones, adopt, and fine-tune

**Architecture (Model II, pretrained backbone).**

* **Backbone.** The torchvision **MobileNetV3-Large** feature extractor with ImageNet weights, loaded through the `weights=` API. It is 3M parameters and fast; `efficientnet_b0` is also supported through `CFG`. Its stride-32 output for a 64×256 input is a 960×2×8 map.
* **Head.** The same `LPRHead` as Model I (1×1 reduction, 1×8 column pooling, FC, LayerNorm, Dropout, 7 digit heads and 1 letter head), with the same initialization.
* **Input.** ImageNet normalization and 3-channel RGB; grayscale input is expanded to 3 channels in `to_pil`.
* **Two-stage fine-tuning:**
  1. **Stage 1:** the backbone is frozen, and its BatchNorm layers are kept in eval mode so the ImageNet statistics are not overwritten. Only the new head trains, for a few epochs at LR 1e-3.
  2. **Stage 2:** everything is unfrozen, and training uses parameter groups with a **lower backbone LR** (3e-4) than the head LR (1e-3). Stage 2 starts from the best stage-1 weights and gets its own OneCycle schedule and early stopping.

In [ ]:
%%writefile "{LPR_PARTS_DIR}/20_pretrained_model.py"


class PretrainedLPRNet(nn.Module):
    """Model II: ImageNet-pretrained torchvision backbone + the shared LPRHead."""

    def __init__(self, n_letters, pretrained=True, backbone="mobilenet_v3_large", dropout=0.3):
        super().__init__()
        if backbone == "mobilenet_v3_large":
            net = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2 if pretrained else None)
            out_ch = 960
        elif backbone == "efficientnet_b0":
            net = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
            out_ch = 1280
        else:
            raise ValueError(backbone)
        self.backbone_name = backbone
        self.features = net.features  # stride 32: [B,3,64,256] -> [B,out_ch,2,8]
        self.head = LPRHead(out_ch, n_letters, mid_ch=256, pool_w=8, dropout=dropout)
        self.backbone_frozen = False

    def set_backbone_trainable(self, trainable):
        self.backbone_frozen = not trainable
        for p in self.features.parameters():
            p.requires_grad = trainable
        self.train(self.training)

    def train(self, mode=True):
        super().train(mode)
        if self.backbone_frozen:
            self.features.eval()  # frozen stage: keep ImageNet BatchNorm statistics
        return self

    def forward(self, x):
        return self.head(self.features(x))


def build_model(arch, n_letters, pretrained=False, backbone="mobilenet_v3_large"):
    """arch: 'scratch' | 'pretrained'. Use pretrained=False when loading a trained state_dict (no download)."""
    if arch == "scratch":
        return ScratchLPRNet(n_letters)
    if arch == "pretrained":
        return PretrainedLPRNet(n_letters, pretrained=pretrained, backbone=backbone)
    raise ValueError(arch)

In [ ]:
lpr_models = build_lpr_module(upto="20_pretrained_model.py")
_pc = CFG["lpr"]["pretrained"]
set_seed()
pretrained_model = lpr_models.PretrainedLPRNet(N_LETTERS, pretrained=True, backbone=_pc["backbone"])
with torch.no_grad():
    _d, _l = pretrained_model.eval()(torch.zeros(2, 3, *LPR_INPUT_HW))
print("output shapes:", tuple(_d.shape), tuple(_l.shape), "| params (total, trainable):", count_params(pretrained_model))

### 2.8 Two-stage fine-tuning of Model II

In [ ]:
# TODO (done): stage 1 = frozen backbone (head only), stage 2 = unfrozen with a lower backbone LR
pretrained_model.set_backbone_trainable(False)
print("stage 1 trainable params:", count_params(pretrained_model)[1])
pre_hist1, pre_best1 = fit(
    pretrained_model, f"pretrained_stage1_{RUN_TAG}",
    param_groups(pretrained_model.head, _pc["stage1_lr"], _pc["weight_decay"]),
    epochs=_pc["stage1_epochs"], patience=_pc["stage1_epochs"],
)

pretrained_model.set_backbone_trainable(True)
print("stage 2 trainable params:", count_params(pretrained_model)[1])
_groups2 = param_groups(pretrained_model.features, _pc["backbone_lr"], _pc["weight_decay"]) + \
    param_groups(pretrained_model.head, _pc["head_lr"], _pc["weight_decay"])
pre_hist2, pretrained_best_path = fit(
    pretrained_model, f"pretrained_stage2_{RUN_TAG}", _groups2, epochs=_pc["epochs"], patience=_pc["patience"],
    init_state=torch.load(pre_best1, map_location="cpu", weights_only=True)["state_dict"],
)
pretrained_history = concat_histories(pre_hist1, pre_hist2)

In [ ]:
save_lpr_model_dir("pretrained", pretrained_model, pretrained_history)  # saved first, before any plotting
plot_history(pretrained_history, "Model II: MobileNetV3 (stage 1 frozen -> stage 2 fine-tune)", "pretrained")
grad_health_report(pretrained_history, "pretrained")

### 2.9 Reading the gradient-health plots

* **No exploding gradients.** The global gradient norm (log scale) stays in a narrow band, with no upward trend or spikes of several orders of magnitude and no NaN/Inf. Clipping at 5 only fires occasionally, mostly during warm-up.
* **No vanishing gradients.** The per-layer norms of the *first* conv layer stay within a few orders of magnitude of the head layers instead of decaying towards 0. Residual connections (Model I) and BatchNorm in both models keep the signal flowing. In Model II's stage 1 the backbone curves are empty, because those layers are frozen on purpose.
* **No saturation or dead units.** The zero-fraction of the monitored ReLUs stays well below 1: about 0.5 is normal for ReLU, and a value near 1 would mean dead units. The share of letter predictions with softmax > 0.99 rises gradually as the model becomes confident, and label smoothing keeps it from locking in at 100% early.
* `grad_health_report` prints these checks as numbers. Re-read them after the full cloud run, since the local smoke test only covers 1 epoch.

### 2.10 Test-set evaluation of both models

In [ ]:
# TODO (done): Calculate accuracy and show mis-classified samples
from sklearn.metrics import classification_report, confusion_matrix, f1_score

LPR_TEST_RESULTS, LPR_TEST_PREDS = {}, {}
_test_ds = lpr_datasets["test"]
for name, model in (("scratch", scratch_model), ("pretrained", pretrained_model)):
    r = evaluate(model, lpr_loaders["test"], return_preds=True)
    P, Y = r.pop("preds"), r.pop("targets")
    LPR_TEST_PREDS[name], LPR_TEST_TARGETS = P, Y
    _pred_df = pd.DataFrame({"filename": _test_ds.files, "true": [decode_target(t) for t in Y],
                             "pred": [decode_target(t) for t in P], "correct": (P == Y).all(1).numpy(),
                             "n_wrong_chars": (P != Y).sum(1).numpy()})
    for i in range(PLATE_LEN):
        _pred_df[f"slot{i}_ok"] = (P[:, i] == Y[:, i]).numpy()
    _pred_df.to_csv(RESULTS_DIR / f"lpr_{name}_test_predictions.csv", index=False, encoding="utf-8")
    r["digit_macro_f1"] = float(f1_score(Y[:, DIGIT_POS].ravel(), P[:, DIGIT_POS].ravel(), labels=list(range(10)),
                                         average="macro", zero_division=0))
    _present = sorted(set(Y[:, LETTER_POS].tolist()))
    r["letter_macro_f1"] = float(f1_score(Y[:, LETTER_POS], P[:, LETTER_POS], labels=_present, average="macro", zero_division=0))
    r["letter_acc"] = float((P[:, LETTER_POS] == Y[:, LETTER_POS]).float().mean())
    LPR_TEST_RESULTS[name] = r
    _rep = classification_report(Y[:, LETTER_POS], P[:, LETTER_POS], labels=_present, output_dict=True, zero_division=0,
                                 target_names=[f"{LETTER_VOCAB[i]} ({LETTER_VOCAB_LATIN[i]})" for i in _present])
    pd.DataFrame(_rep).T.to_csv(RESULTS_DIR / f"lpr_{name}_letter_report.csv", encoding="utf-8")
    log(f"\n=== LPR {name} test: plate acc {r['plate_acc']:.4f} | char acc {r['char_acc']:.4f} | loss {r['loss']:.4f} | "
          f"macro F1 digits {r['digit_macro_f1']:.4f} letter {r['letter_macro_f1']:.4f}")
    display(pd.DataFrame([r["pos_acc"]], columns=[f"slot{i}{'(L)' if i == LETTER_POS else ''}" for i in range(PLATE_LEN)],
                         index=["accuracy"]).round(4))

    _cls = sorted(set(_present) | set(P[:, LETTER_POS].tolist()))
    cm = confusion_matrix(Y[:, LETTER_POS], P[:, LETTER_POS], labels=_cls)
    plt.figure(figsize=(13, 11))
    plt.imshow(cm / np.maximum(cm.sum(1, keepdims=True), 1), cmap="Blues", vmin=0, vmax=1)
    plt.colorbar(label="fraction of the true class")
    for (ti, pi), v in np.ndenumerate(cm):
        if v:
            plt.text(pi, ti, str(v), ha="center", va="center", fontsize=9,
                     color="white" if v / max(cm[ti].sum(), 1) > 0.5 else "black")
    plt.xticks(range(len(_cls)), [LETTER_VOCAB_LATIN[i] for i in _cls], rotation=70)
    plt.yticks(range(len(_cls)), [f"{LETTER_VOCAB_LATIN[i]} ({cm[k].sum()})" for k, i in enumerate(_cls)])
    plt.title(f"{name}: letter confusion on the test set (counts in cells)")
    plt.xlabel("predicted letter")
    plt.ylabel("true letter (test count)")
    plt.grid(False)
    show(f"lpr_{name}_letter_confusion")

    wrong = torch.nonzero(~(P == Y).all(1)).ravel().tolist()
    if wrong:
        fig, axes = plt.subplots(6, 2, figsize=(16, 16))
        for a in axes.ravel():
            a.axis("off")
        for a, i in zip(axes.ravel(), wrong[:12]):
            a.imshow(Image.open(_test_ds.img_dir / _test_ds.files[i]))
            a.set_title(f"true {plate_to_latin(decode_target(Y[i]))}   pred {plate_to_latin(decode_target(P[i]))}", fontsize=13)
        plt.suptitle(f"{name}: {len(wrong)} misclassified test plates ({len(wrong) / len(Y):.1%}), first 12", fontsize=18)
        show(f"lpr_{name}_misclassified")

In [ ]:
pd.DataFrame({n: LPR_TEST_RESULTS[n]["pos_acc"] for n in LPR_TEST_RESULTS},
             index=[f"slot{i}{' (letter)' if i == LETTER_POS else ''}" for i in range(PLATE_LEN)]
             ).rename_axis("position").to_csv(RESULTS_DIR / "lpr_per_position_accuracy.csv")

### 2.10b The two recognizers side by side

* Training curves of both models on one plot.
* Test accuracy **per character slot**: which positions are hard?
* Test full-plate accuracy **by crop height**: tiny crops from distant cars are the hard case.
* Test accuracy **per letter class**, including the rare letters.

In [ ]:
_hists = {"scratch": scratch_history, "pretrained": pretrained_history}
_colors = {"scratch": "tab:blue", "pretrained": "tab:orange"}
_test_df = LPR_SPLIT_DFS["test"]
_wd = 0.38
plt.figure()
for n, h in _hists.items():
    plt.plot(h["epoch"], h["val_loss"], "o-", color=_colors[n], label=f"{n} val")
    plt.plot(h["epoch"], h["train_loss"], "--", color=_colors[n], alpha=0.6, label=f"{n} train")
plt.title("both recognizers: loss per epoch")
plt.xlabel("epoch")
plt.ylabel("loss (train: weighted+smoothed CE, val: plain CE)")
plt.legend()
show("lpr_both_loss")

plt.figure()
for n, h in _hists.items():
    plt.plot(h["epoch"], h["val_plate_acc"], "o-", color=_colors[n], label=f"{n} full-plate")
    plt.plot(h["epoch"], h["val_char_acc"], "s--", color=_colors[n], alpha=0.6, label=f"{n} per-char")
plt.title("both recognizers: validation accuracy per epoch")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.ylim(0, 1.02)
plt.legend()
show("lpr_both_val_accuracy")

_slots = np.arange(PLATE_LEN)
plt.figure()
for k, n in enumerate(_hists):
    plt.bar(_slots + (k - 0.5) * _wd, LPR_TEST_RESULTS[n]["pos_acc"], _wd, color=_colors[n], label=n)
plt.xticks(_slots, [f"slot {i}\n{'letter' if i == LETTER_POS else 'digit'}" for i in _slots])
plt.title("test accuracy per character slot")
plt.ylabel("accuracy")
plt.ylim(0, 1.05)
plt.legend()
show("lpr_both_per_slot_accuracy")

_hb = [0, 25, 40, 60, 100, 200, 1e5]
_hl = ["<25", "25-40", "40-60", "60-100", "100-200", ">200"]
_hc = pd.cut(_test_df.h, _hb, labels=_hl)
_cnt = _hc.value_counts().reindex(_hl).fillna(0).astype(int)
_rows = []
plt.figure()
for k, n in enumerate(_hists):
    _ok = pd.Series((LPR_TEST_PREDS[n] == LPR_TEST_TARGETS).all(1).numpy())
    _acc = _ok.groupby(_hc, observed=False).mean().reindex(_hl).fillna(0)
    plt.bar(np.arange(len(_hl)) + (k - 0.5) * _wd, _acc.values, _wd, color=_colors[n], label=n)
    _rows += [{"model": n, "crop_height_px": l, "n_test": int(c), "plate_acc": float(v)} for l, c, v in zip(_hl, _cnt, _acc.values)]
plt.xticks(range(len(_hl)), [f"{l} px\n(n={c})" for l, c in zip(_hl, _cnt)])
plt.title("test full-plate accuracy by crop height")
plt.ylabel("full-plate accuracy")
plt.ylim(0, 1.05)
plt.legend()
show("lpr_both_accuracy_by_crop_height")
pd.DataFrame(_rows).to_csv(RESULTS_DIR / "lpr_accuracy_by_crop_height.csv", index=False)

_Yl = LPR_TEST_TARGETS[:, LETTER_POS].numpy()
_present = sorted(set(_Yl.tolist()))
_rows = []
fig, a = plt.subplots(figsize=(16, 7))
for k, n in enumerate(_hists):
    _Pl = LPR_TEST_PREDS[n][:, LETTER_POS].numpy()
    _acc = [float((_Pl[_Yl == c] == c).mean()) for c in _present]
    a.bar(np.arange(len(_present)) + (k - 0.5) * _wd, _acc, _wd, color=_colors[n], label=n)
    _rows += [{"model": n, "letter": LETTER_VOCAB[c], "latin": LETTER_VOCAB_LATIN[c], "n_test": int((_Yl == c).sum()),
               "accuracy": v} for c, v in zip(_present, _acc)]
a.set_xticks(range(len(_present)), [f"{LETTER_VOCAB_LATIN[c]}\n(n={int((_Yl == c).sum())})" for c in _present], fontsize=11)
a.set(title="test accuracy per letter class", ylabel="accuracy", ylim=(0, 1.05))
a.legend()
show("lpr_both_per_letter_accuracy")
pd.DataFrame(_rows).to_csv(RESULTS_DIR / "lpr_per_letter_accuracy.csv", index=False, encoding="utf-8")

#### Compare two models

### 2.11 Comparison and choice of the best model

Timings use warm-up iterations and device synchronization (`torch.cuda.synchronize()` or `torch.mps.synchronize()`). They are reported on the accelerator and on the CPU, per image (batch 1) and per batch.

In [ ]:
# TODO (done): compare accuracy, capacity, loss and inference time
_tcfg = CFG["lpr"]["timing"]


def time_inference(model, dev, batch):
    model = model.to(dev).eval()
    x = torch.randn(batch, 3, *LPR_INPUT_HW, device=dev)
    with torch.no_grad():
        for _ in range(_tcfg["warmup"]):
            model(x)
        sync_device(dev)
        t0 = time.perf_counter()
        for _ in range(_tcfg["iters"]):
            model(x)
        sync_device(dev)
    return (time.perf_counter() - t0) / _tcfg["iters"] * 1000  # ms per forward


_meta = {"n_letters": N_LETTERS, "letter_vocab": LETTER_VOCAB, "input_size": list(LPR_INPUT_HW), "backbone": _pc["backbone"]}
_rows = []
for name, model in (("scratch", scratch_model), ("pretrained", pretrained_model)):
    ck_path = ARTIFACT_DIR / f"lpr_{name}_best.pt"
    torch.save({"state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()}, "arch": name, **_meta}, ck_path)
    total, trainable = count_params(model)
    r = LPR_TEST_RESULTS[name]
    row = {"model": name, "test_plate_acc": r["plate_acc"], "test_char_acc": r["char_acc"], "test_letter_acc": r["letter_acc"],
           "digit_macro_f1": r["digit_macro_f1"], "letter_macro_f1": r["letter_macro_f1"], "test_loss": r["loss"],
           "params_total": total, "params_trainable": trainable, "size_MB": ck_path.stat().st_size / 2**20}
    cpu_copy = copy.deepcopy(model).cpu()
    if DEVICE.type != "cpu":
        row[f"{DEVICE.type}_ms_per_image"] = time_inference(model, DEVICE, 1)
        row[f"{DEVICE.type}_ms_per_batch{_tcfg['batch']}"] = time_inference(model, DEVICE, _tcfg["batch"])
    row["cpu_ms_per_image"] = time_inference(cpu_copy, torch.device("cpu"), 1)
    row[f"cpu_ms_per_batch{_tcfg['batch']}"] = time_inference(cpu_copy, torch.device("cpu"), _tcfg["batch"])
    model.to(DEVICE)
    _rows.append(row)
LPR_COMPARISON = pd.DataFrame(_rows).set_index("model")
LPR_COMPARISON.to_csv(ARTIFACT_DIR / "lpr_comparison.csv")
display(LPR_COMPARISON.T)

_speed_col = f"{DEVICE.type}_ms_per_image" if DEVICE.type != "cpu" else "cpu_ms_per_image"
BEST_LPR_NAME = LPR_COMPARISON.sort_values(["test_plate_acc", _speed_col], ascending=[False, True]).index[0]
log("LPR comparison:\n" + LPR_COMPARISON.T.to_string())
log(f"best model: {BEST_LPR_NAME} (highest test full-plate accuracy, ties broken by {_speed_col})")
_cmp = [("test_plate_acc", "test full-plate acc"), ("test_char_acc", "test per-char acc"), ("letter_macro_f1", "letter macro F1"),
        ("test_loss", "test loss"), ("params_total", "parameters"), ("size_MB", "checkpoint MB"), (_speed_col, f"{_speed_col} (ms)")]
if _speed_col != "cpu_ms_per_image":
    _cmp.append(("cpu_ms_per_image", "cpu_ms_per_image (ms)"))
for c, t in _cmp:
    v = LPR_COMPARISON[c]
    plt.figure(figsize=(7, 6))
    plt.bar(v.index, v.values, color=["tab:blue" if m == "scratch" else "tab:orange" for m in v.index], width=0.55)
    for i, x in enumerate(v.values):
        plt.text(i, x, f"{x:.4g}", ha="center", va="bottom", fontsize=13)
    plt.title(f"{t} (best model = {BEST_LPR_NAME})")
    show(f"lpr_compare_{c}")
LPR_METRICS = {"smoke_test": SMOKE_TEST, "best_model": BEST_LPR_NAME, "selection": "max test_plate_acc, tie-break min latency",
               "n_samples": {s: len(d) for s, d in LPR_SPLIT_DFS.items()}, "test": LPR_TEST_RESULTS,
               "comparison": json.loads(LPR_COMPARISON.to_json(orient="index"))}
(ARTIFACT_DIR / "lpr_metrics.json").write_text(json.dumps(LPR_METRICS, indent=2, ensure_ascii=False), encoding="utf-8")
for _name, _d in LPR_MODEL_DIRS.items():
    (_d / "test_metrics.json").write_text(json.dumps({**LPR_TEST_RESULTS[_name], **json.loads(LPR_COMPARISON.loc[[_name]].to_json(orient="records"))[0],
                                                      "is_best_model": _name == BEST_LPR_NAME}, indent=2), encoding="utf-8")
    shutil.copy2(RESULTS_DIR / f"lpr_{_name}_test_predictions.csv", _d / "test_predictions.csv")

## Evaluation on train, validation and test

Every model is scored on **all three splits** with the same metrics. Reading the numbers side by side shows how well it generalizes:

* **train ≈ validation ≈ test:** the model generalizes well.
* **train much higher than validation/test:** overfitting (it memorized the training data).
* **all three low:** underfitting (the model or the training is too weak).
* **validation ≈ test:** model selection (early stopping, best checkpoint) did not overfit the validation split. Test is the honest final number.

The recognizers are scored on the training images **without augmentation**, exactly like validation and test. The training loss/accuracy printed during training used augmented images, so it is not comparable.

Outputs: `outputs/split_metrics_detector.csv` and `outputs/split_metrics_recognizers.csv`, plus the figures below.

In [ ]:
# ---------------------------------------------------------------- detector: same metrics on train / val / test
free_gpu()
_det_rows = []
for _split in ("train", "val", "test"):
    if _split == "val":
        _m = lpd_val_metrics
    elif _split == "test":
        _m = lpd_test_metrics
    else:
        _m = lpd_model.val(split="train", name=f"{LPD_RUN_NAME}_train", **{**_val_kw, "plots": False})
    _det_rows.append({"split": _split, "images": len(LPD_USE[_split]),
                      **{k: v for k, v in det_metrics(_m).items() if k != "speed_ms_per_image"}})
free_gpu()
SPLIT_METRICS_DET = pd.DataFrame(_det_rows).set_index("split")
SPLIT_METRICS_DET.to_csv(RESULTS_DIR / "split_metrics_detector.csv")
SPLIT_METRICS_DET.to_csv(LPD_MODEL_DIR / "metrics_train_val_test.csv")
log("detector metrics per split:\n" + SPLIT_METRICS_DET.round(4).to_string())
display(SPLIT_METRICS_DET.round(4))

_split_colors = {"train": "tab:blue", "val": "tab:orange", "test": "tab:green"}
_det_cols = ["precision", "recall", "mAP50", "mAP50-95"]
_x = np.arange(len(_det_cols))
plt.figure()
for k, _split in enumerate(SPLIT_METRICS_DET.index):
    _v = SPLIT_METRICS_DET.loc[_split, _det_cols].to_numpy(float)
    plt.bar(_x + (k - 1) * 0.27, _v, 0.27, color=_split_colors[_split], label=f"{_split} ({SPLIT_METRICS_DET.loc[_split, 'images']} img)")
    for xi, vi in zip(_x + (k - 1) * 0.27, _v):
        plt.text(xi, vi, f"{vi:.3f}", ha="center", va="bottom", fontsize=10)
plt.xticks(_x, _det_cols)
plt.ylim(0, 1.1)
plt.ylabel("score")
plt.title("YOLO26 detector: train vs validation vs test")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3)
show("eval_detector_train_val_test")

# ---------------------------------------------------------------- recognizers: same metrics on train / val / test
_eval_sets = {"train": PLPRDataset(LPR_SPLIT_DFS["train"], LPR_DIR / "detections", augment=None),  # no augmentation
              "val": lpr_datasets["val"], "test": lpr_datasets["test"]}
_eval_loaders = {s: make_loader(ds, train=False) for s, ds in _eval_sets.items()}
_rec_rows = []
for _name, _model in (("scratch", scratch_model), ("pretrained", pretrained_model)):
    for _split, _loader in _eval_loaders.items():
        _r = evaluate(_model, _loader, return_preds=True)
        _P, _Y = _r.pop("preds"), _r.pop("targets")
        _rec_rows.append({
            "model": _name, "split": _split, "samples": len(_Y), "plate_acc": _r["plate_acc"], "char_acc": _r["char_acc"],
            "letter_acc": float((_P[:, LETTER_POS] == _Y[:, LETTER_POS]).float().mean()),
            "digit_macro_f1": float(f1_score(_Y[:, DIGIT_POS].ravel(), _P[:, DIGIT_POS].ravel(), labels=list(range(10)),
                                             average="macro", zero_division=0)),
            "letter_macro_f1": float(f1_score(_Y[:, LETTER_POS], _P[:, LETTER_POS], labels=sorted(set(_Y[:, LETTER_POS].tolist())),
                                              average="macro", zero_division=0)),
            "loss": _r["loss"]})
    free_gpu()
SPLIT_METRICS_REC = pd.DataFrame(_rec_rows).set_index(["model", "split"])
SPLIT_METRICS_REC.to_csv(RESULTS_DIR / "split_metrics_recognizers.csv")
log("recognizer metrics per split:\n" + SPLIT_METRICS_REC.round(4).to_string())
for _name, _d in LPR_MODEL_DIRS.items():
    SPLIT_METRICS_REC.loc[_name].to_csv(_d / "metrics_train_val_test.csv")
display(SPLIT_METRICS_REC.round(4))

_rec_cols = ["plate_acc", "char_acc", "letter_acc", "digit_macro_f1", "letter_macro_f1"]
_x = np.arange(len(_rec_cols))
for _name in ("scratch", "pretrained"):
    _t = SPLIT_METRICS_REC.loc[_name]
    plt.figure(figsize=(13, 6.5))
    for k, _split in enumerate(_t.index):
        _v = _t.loc[_split, _rec_cols].to_numpy(float)
        plt.bar(_x + (k - 1) * 0.27, _v, 0.27, color=_split_colors[_split], label=f"{_split} ({_t.loc[_split, 'samples']} crops)")
        for xi, vi in zip(_x + (k - 1) * 0.27, _v):
            plt.text(xi, vi, f"{vi:.3f}", ha="center", va="bottom", fontsize=9)
    plt.xticks(_x, ["full-plate acc", "per-char acc", "letter acc", "digit macro F1", "letter macro F1"])
    plt.ylim(0, 1.1)
    plt.ylabel("score")
    plt.title(f"{_name} recognizer: train vs validation vs test")
    plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3)
    show(f"eval_{_name}_train_val_test")

plt.figure()
for k, _name in enumerate(("scratch", "pretrained")):
    _v = SPLIT_METRICS_REC.loc[_name, "loss"].reindex(["train", "val", "test"]).to_numpy(float)
    plt.bar(np.arange(3) + (k - 0.5) * 0.38, _v, 0.38, color=["tab:blue", "tab:orange"][k], label=_name)
    for xi, vi in zip(np.arange(3) + (k - 0.5) * 0.38, _v):
        plt.text(xi, vi, f"{vi:.3f}", ha="center", va="bottom", fontsize=10)
plt.xticks(range(3), ["train", "val", "test"])
plt.ylabel("cross-entropy loss (plain)")
plt.title("recognizer loss per split")
plt.legend()
show("eval_recognizers_loss_per_split")


# ---------------------------------------------------------------- automatic reading of the gaps
def _verdict(train, val, test, what):
    gap = train - test
    if max(train, val, test) < 0.5:
        v = "UNDERFITTING (all splits low)"
    elif gap > 0.05:
        v = f"possible OVERFITTING (train is {gap:.1%} above test)"
    else:
        v = f"generalizes well (train-test gap {gap:+.1%})"
    vt = "" if abs(val - test) <= 0.03 else f"; note val and test differ by {val - test:+.1%}"
    log(f"{what}: train {train:.4f} | val {val:.4f} | test {test:.4f} -> {v}{vt}")


_d = SPLIT_METRICS_DET["mAP50"]
_verdict(_d["train"], _d["val"], _d["test"], "detector mAP50")
for _name in ("scratch", "pretrained"):
    _a = SPLIT_METRICS_REC.loc[_name, "plate_acc"]
    _verdict(_a["train"], _a["val"], _a["test"], f"{_name} full-plate accuracy")

## Export for Phase 3

This section writes a self-contained bundle to `ARTIFACT_DIR` and zips it to `lpd_lpr_artifacts.zip`, which is copied to Drive when `USE_DRIVE` is on. Phase 3 needs **only this zip**: no retraining and no notebook variables.

| file | content |
|---|---|
| `lpd_best.pt` | trained YOLO26 detector |
| `lpd_data.yaml`, `lpd_*_split.txt`, `lpd_metrics.json` | detector data config, splits (the test split is held out), metrics |
| `lpr_scratch_best.pt`, `lpr_pretrained_best.pt` | LPR `state_dict` checkpoints (loadable with `weights_only=True`) |
| `lpr_best.ts` | best recognizer as TorchScript (runs without the class definitions) |
| `lpr_models.py` | model classes, `preprocess`, `crop_with_padding`, `decode` |
| `lpr_config.json` | vocabularies, input size, normalization, layout, crop padding, YOLO threshold |
| `lpr_comparison.csv`, `lpr_metrics.json`, `lpr_*_split.csv` | LPR results and splits |
| `outputs/` | all CSV result tables, the text run log and every figure as PNG (see the last cell) |
| `MANIFEST.md` | description of every file, key metrics, loading snippet |
| `plate_reader.py`, `requirements.txt`, `APP_README.md`, optional `*.onnx` | ready-to-use inference package for apps (added in the next section) |

In [ ]:
# 1) TorchScript export of the best recognizer (traced on CPU, checked against eager)
_best_ck = torch.load(ARTIFACT_DIR / f"lpr_{BEST_LPR_NAME}_best.pt", map_location="cpu", weights_only=True)
_best_cpu = lpr_models.build_model(BEST_LPR_NAME, N_LETTERS, pretrained=False, backbone=_pc["backbone"])
_best_cpu.load_state_dict(_best_ck["state_dict"])
_best_cpu.eval()
_xs = torch.stack([lpr_datasets["test"][i][0] for i in range(min(8, len(lpr_datasets["test"])))])
with torch.no_grad():
    _ts = torch.jit.trace(_best_cpu, torch.zeros(1, 3, *LPR_INPUT_HW))
    _ts.save(str(ARTIFACT_DIR / "lpr_best.ts"))
    _ts = torch.jit.load(str(ARTIFACT_DIR / "lpr_best.ts"))
    _eager, _traced = _best_cpu(_xs), _ts(_xs)
_max_diff = max((a - b).abs().max().item() for a, b in zip(_eager, _traced))
assert _max_diff < 1e-4, _max_diff
assert lpr_models.decode_batch(_eager, LETTER_VOCAB) == lpr_models.decode_batch(_traced, LETTER_VOCAB)
print(f"TorchScript OK: max |eager - traced| = {_max_diff:.2e} on a batch of {len(_xs)} test plates")

# 2) supporting files
shutil.copy2(LPR_MODULE_PATH, ARTIFACT_DIR / "lpr_models.py")
for _d in LPR_MODEL_DIRS.values():  # final module (has both model classes + build_model)
    shutil.copy2(LPR_MODULE_PATH, _d / "lpr_models.py")
shutil.copy2(LPD_DATA_YAML, ARTIFACT_DIR / "lpd_data.yaml")
LPR_CONFIG = {
    "best_model": BEST_LPR_NAME,
    "arch": BEST_LPR_NAME,
    "backbone": _pc["backbone"] if BEST_LPR_NAME == "pretrained" else None,
    "torchscript": "lpr_best.ts",
    "state_dict": f"lpr_{BEST_LPR_NAME}_best.pt",
    "input_size_hw": list(LPR_INPUT_HW),
    "resize": "letterbox (keep aspect ratio, centre, pad with gray)",
    "pad_value": lpr_models.PAD_VALUE,
    "color_mode": "RGB",
    "normalization": {"mean": list(lpr_models.MEAN), "std": list(lpr_models.STD)},
    "digit_vocab": DIGIT_VOCAB,
    "letter_vocab": LETTER_VOCAB,
    "letter_vocab_latin": LETTER_VOCAB_LATIN,
    "letter_index": [{"index": i, "char": c, "latin": l} for i, (c, l) in enumerate(zip(LETTER_VOCAB, LETTER_VOCAB_LATIN))],
    "plate_len": PLATE_LEN,
    "position_layout": ["letter" if i == LETTER_POS else "digit" for i in range(PLATE_LEN)],
    "letter_position": LETTER_POS,
    "digit_positions": DIGIT_POS,
    "model_outputs": "tuple(digit_logits [B,7,10] for label positions 0,1,3,4,5,6,7; letter_logits [B,n_letters])",
    "output_format": "DD L DDD DD without separators, e.g. '12ب34567' (last two digits = region code; alef is written 'الف')",
    "label_normalization": "lpr_models.normalize_label (removes ZWJ/ZWNJ/spaces, Persian digits -> ASCII, Arabic yeh/kaf -> Persian)",
    "crop_padding": 0.05,
    "yolo_conf_threshold": round(float(LPD_CONF_THRESHOLD), 3),
    "yolo_imgsz": LPD_IMGSZ,
    "smoke_test": SMOKE_TEST,
}
(ARTIFACT_DIR / "lpr_config.json").write_text(json.dumps(LPR_CONFIG, indent=2, ensure_ascii=False), encoding="utf-8")

# 3) MANIFEST.md
_c = LPR_COMPARISON
_manifest = f"""# LPD + LPR artifact bundle (Phase 1 & 2 -> Phase 3)

Produced by `License_Plate_Detection_Recognition_cloud.ipynb` on **{PLATFORM}** ({DEVICE}), {time.strftime('%Y-%m-%d %H:%M')}.
{'**SMOKE TEST BUNDLE: models trained for 1 epoch on a few dozen samples. Metrics are meaningless.**' if SMOKE_TEST else ''}

## Files
| file | what | how it was produced |
|---|---|---|
| `lpd_best.pt` | YOLO26 plate detector (1 class `license_plate`) | Ultralytics fine-tune of `{LPD_CFG['model']}`, imgsz {LPD_IMGSZ}, best epoch by val fitness |
| `lpd_data.yaml` | dataset yaml used for training (`path` is platform specific) | notebook 1.4 |
| `lpd_train/val/test_split.txt` | LPD image file names per split; **test is held out** (never used for training/selection) | grouped-by-plate 70/15/15 split, seed {SEED} |
| `lpd_metrics.json` | val + test P / R / mAP50 / mAP50-95, recommended conf threshold | `model.val(split=...)` |
| `lpr_scratch_best.pt`, `lpr_pretrained_best.pt` | `{{"state_dict", "arch", "n_letters", "letter_vocab", "input_size", "backbone"}}`, loadable with `torch.load(..., weights_only=True)` | best val full-plate accuracy |
| `lpr_best.ts` | TorchScript of the best recognizer (**{BEST_LPR_NAME}**), traced on CPU and verified against eager | notebook export cell |
| `lpr_models.py` | model classes, `preprocess`, `crop_with_padding`, `normalize_label`, `decode`, `decode_batch` (torch/torchvision/numpy/PIL only) | same file the notebook trained with |
| `lpr_config.json` | vocabularies (Persian + Latin), input size, normalization, layout, crop padding, YOLO threshold | notebook export cell |
| `plate_reader.py`, `requirements.txt`, `APP_README.md`, optional `lpd_best.onnx` / `lpr_best.onnx` | **inference package for apps**: `PlateReader(...).read(image)` returns plate text + boxes; also a command-line tool | notebook section *Inference package for an app* |
| `lpr_comparison.csv`, `lpr_metrics.json` | test metrics, params, size, latency of both recognizers | notebook 2.10-2.11 |
| `outputs/*.csv`, `outputs/figures/*.png`, `outputs/run_log_*.txt` | per-epoch histories, per-sample test predictions, per-letter / per-slot / per-size accuracy, detector error analysis, all figures, text log | written throughout the notebook |
| `lpr_train/val/test_split.csv` | LPR splits (`filename,label`); LPR plates that appear in LPD-test images are all in `lpr_test` | grouped-by-plate, letter-stratified |

## Key metrics
* Detector ({LPD_CFG['model']}): test mAP50 **{LPD_METRICS['test']['mAP50']:.4f}**, mAP50-95 **{LPD_METRICS['test']['mAP50-95']:.4f}**, P {LPD_METRICS['test']['precision']:.4f}, R {LPD_METRICS['test']['recall']:.4f}; recommended conf **{LPR_CONFIG['yolo_conf_threshold']}**
* Recognizer test full-plate / per-char accuracy: scratch **{_c.loc['scratch', 'test_plate_acc']:.4f}** / {_c.loc['scratch', 'test_char_acc']:.4f}, pretrained **{_c.loc['pretrained', 'test_plate_acc']:.4f}** / {_c.loc['pretrained', 'test_char_acc']:.4f} -> best = **{BEST_LPR_NAME}**

## Notes for Phase 3
* **LPD annotations include plate text: {'YES' if LPD_HAS_PLATE_TEXT else 'NO'}.** `plate_labels.csv` (in the LPD dataset, not in this bundle) has one row per image; multiple plates are space-separated. Their order relative to the boxes in the label .txt is not documented, so match per image (e.g. by string). Normalize ground truth with `lpr_models.normalize_label` before comparing.
* Recognizer input: RGB crop -> `preprocess()` (letterbox 64x256, ImageNet norm). For cv2 (BGR) arrays use `preprocess(crop, bgr=True)`.
* Crop the YOLO box enlarged by `crop_padding` = {LPR_CONFIG['crop_padding']} of its size per side (`crop_with_padding`); training used +/-8% crop jitter.

## Load and read one plate
```python
import json, cv2, torch, lpr_models as M            # run inside the extracted bundle folder
from ultralytics import YOLO
cfg = json.load(open("lpr_config.json", encoding="utf-8"))
det, rec = YOLO("lpd_best.pt"), torch.jit.load("lpr_best.ts").eval()
img = cv2.imread("car.jpg")                                             # BGR
boxes = det.predict(img, conf=cfg["yolo_conf_threshold"], imgsz=cfg["yolo_imgsz"])[0].boxes
xyxy = boxes.xyxy[boxes.conf.argmax()].tolist()                         # most confident plate
crop = M.crop_with_padding(img, xyxy, cfg["crop_padding"])
with torch.no_grad(): out = rec(M.preprocess(crop, bgr=True)[None])
print(M.decode(out, cfg["letter_vocab"]))                               # e.g. '12ب34567'
```
"""
(ARTIFACT_DIR / "MANIFEST.md").write_text(_manifest, encoding="utf-8")

# 4) zip (+ copy to Drive)
BUNDLE_FILES = ["lpd_best.pt", "lpd_data.yaml", "lpd_train_split.txt", "lpd_val_split.txt", "lpd_test_split.txt",
                "lpd_metrics.json", "lpr_scratch_best.pt", "lpr_pretrained_best.pt", "lpr_best.ts", "lpr_models.py",
                "lpr_config.json", "lpr_comparison.csv", "lpr_metrics.json", "lpr_train_split.csv", "lpr_val_split.csv",
                "lpr_test_split.csv", "MANIFEST.md"]
BUNDLE_ZIP = OUT_DIR / "lpd_lpr_artifacts.zip"


def write_bundle_zip():
    """(Re)write the bundle zip: model/config/app files + everything in outputs/ (CSVs, figures, run log)."""
    extra = [f for f in sorted(RESULTS_DIR.rglob("*")) if f.is_file()]
    with zipfile.ZipFile(BUNDLE_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in BUNDLE_FILES:
            zf.write(ARTIFACT_DIR / f, arcname=f"lpd_lpr_artifacts/{f}")
        for f in extra:
            zf.write(f, arcname=f"lpd_lpr_artifacts/outputs/{f.relative_to(RESULTS_DIR).as_posix()}")
    log(f"{BUNDLE_ZIP}  ({BUNDLE_ZIP.stat().st_size / 2**20:.1f} MB, {len(BUNDLE_FILES)} files + {len(extra)} result files)")
    if PERSIST_DIR != OUT_DIR:
        shutil.copy2(BUNDLE_ZIP, PERSIST_DIR / BUNDLE_ZIP.name)
        print("copied to", PERSIST_DIR / BUNDLE_ZIP.name)


write_bundle_zip()

## Inference package for an app

The bundle also contains a ready-to-use **plate reader for your own applications**.

| file | what it is |
|---|---|
| `plate_reader.py` | `PlateReader` class and command-line tool: image → list of plates with box, detector confidence, plate text (Persian + Latin) and a text confidence |
| `requirements.txt` | what the app needs: torch, torchvision, ultralytics, opencv-python, numpy, pillow |
| `APP_README.md` | how to use it (Python API, command line, JSON output, a small web-API example) |
| `lpd_best.onnx`, `lpr_best.onnx` | *optional* ONNX versions of the detector and recognizer for apps without PyTorch (onnxruntime, mobile, web). They are only included if the export succeeds. |

Use in Python, after unzipping `lpd_lpr_artifacts.zip`:
```python
from plate_reader import PlateReader
reader = PlateReader("lpd_lpr_artifacts")
plates = reader.read("car.jpg")   # [{'text': '12ب34567', 'latin': '12-B-345-67', 'det_confidence': 0.93, 'text_confidence': 0.99, 'box': [x1, y1, x2, y2]}, ...]
```
Or from the command line: `python lpd_lpr_artifacts/plate_reader.py car.jpg --json --save-vis out/`

In [ ]:
%%writefile "{ARTIFACT_DIR}/plate_reader.py"
"""plate_reader.py - Iranian licence plate reader (YOLO26 detector + CNN recognizer) for applications.

Python:
    from plate_reader import PlateReader
    reader = PlateReader("path/to/lpd_lpr_artifacts")      # folder that contains this file
    for p in reader.read("car.jpg"):                      # path, PIL image, or BGR numpy array (cv2)
        print(p["text"], p["latin"], p["det_confidence"], p["text_confidence"], p["box"])
    text, conf = reader.read_crop(plate_crop_bgr)          # recognizer only, for an already-cropped plate

Command line:
    python plate_reader.py car1.jpg car2.jpg [--json] [--save-vis OUT_DIR] [--device cpu|cuda|mps] [--conf 0.3]
    python plate_reader.py plate_crop.jpg --crop           # image is already a plate crop

Requirements: torch, torchvision, ultralytics>=8.4.0, opencv-python, numpy, pillow (see requirements.txt).
"""
import argparse
import json
import sys
from pathlib import Path

import cv2
import numpy as np
import torch

HERE = Path(__file__).resolve().parent
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))
import lpr_models as M  # noqa: E402  (shipped next to this file)


class PlateReader:
    def __init__(self, bundle_dir=HERE, device=None, conf=None, max_plates=None):
        from ultralytics import YOLO

        b = Path(bundle_dir)
        self.cfg = json.loads((b / "lpr_config.json").read_text(encoding="utf-8"))
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        self.conf = float(self.cfg["yolo_conf_threshold"] if conf is None else conf)
        self.max_plates = max_plates
        self.detector = YOLO(str(b / "lpd_best.pt"))
        self.recognizer = torch.jit.load(str(b / "lpr_best.ts"), map_location=self.device).eval()
        self.letters = self.cfg["letter_vocab"]
        self.latin = dict(zip(self.cfg["letter_vocab"], self.cfg["letter_vocab_latin"]))

    @staticmethod
    def _to_bgr(image):
        if isinstance(image, (str, Path)):
            img = cv2.imread(str(image))
            if img is None:
                raise FileNotFoundError(image)
            return img
        if hasattr(image, "convert"):  # PIL image
            return cv2.cvtColor(np.asarray(image.convert("RGB")), cv2.COLOR_RGB2BGR)
        return np.asarray(image)  # assume BGR numpy array (cv2 convention)

    def to_latin(self, text):
        if len(text) < 8:
            return text
        letter = text[2:-5]
        return f"{text[:2]}-{self.latin.get(letter, '?')}-{text[-5:-2]}-{text[-2:]}"

    @torch.no_grad()
    def read_crop(self, crop):
        """Recognize an already-cropped plate. Returns (text, confidence = min per-character probability)."""
        x = M.preprocess(self._to_bgr(crop), bgr=True)[None].to(self.device)
        digit_logits, letter_logits = self.recognizer(x)
        digit_logits, letter_logits = digit_logits.float().cpu(), letter_logits.float().cpu()
        probs = torch.cat([digit_logits.softmax(-1).amax(-1).ravel(), letter_logits.softmax(-1).amax(-1).ravel()])
        return M.decode((digit_logits, letter_logits), self.letters), float(probs.min())

    def read(self, image):
        """Detect and read every plate in a full image, most confident detection first."""
        img = self._to_bgr(image)
        res = self.detector.predict(img, conf=self.conf, imgsz=self.cfg["yolo_imgsz"], device=self.device, verbose=False)[0]
        dets = sorted(zip(res.boxes.xyxy.tolist(), res.boxes.conf.tolist()), key=lambda t: -t[1])[: self.max_plates]
        plates = []
        for box, det_conf in dets:
            crop = M.crop_with_padding(img, box, self.cfg["crop_padding"])
            if crop.size == 0:
                continue
            text, text_conf = self.read_crop(crop)
            plates.append({"text": text, "latin": self.to_latin(text), "det_confidence": round(det_conf, 4),
                           "text_confidence": round(text_conf, 4), "box": [round(v, 1) for v in box]})
        return plates

    def draw(self, image, plates):
        """Return a BGR copy with boxes and Latin plate text (cv2 cannot render Persian script)."""
        img = self._to_bgr(image).copy()
        t = max(2, img.shape[1] // 400)
        for p in plates:
            x1, y1, x2, y2 = map(int, p["box"])
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), t)
            cv2.putText(img, f"{p['latin']} {p['text_confidence']:.2f}", (x1, max(15, y1 - 8)),
                        cv2.FONT_HERSHEY_SIMPLEX, max(0.6, img.shape[1] / 1400), (0, 0, 255), t)
        return img


def main():
    ap = argparse.ArgumentParser(description="Read Iranian licence plates from images.")
    ap.add_argument("images", nargs="+")
    ap.add_argument("--bundle", default=str(HERE), help="folder with lpd_best.pt, lpr_best.ts, lpr_config.json")
    ap.add_argument("--device", default=None)
    ap.add_argument("--conf", type=float, default=None, help="detector confidence threshold (default from config)")
    ap.add_argument("--crop", action="store_true", help="images are already plate crops: skip detection")
    ap.add_argument("--json", action="store_true", help="print one JSON object per image")
    ap.add_argument("--save-vis", default=None, help="folder for annotated images")
    a = ap.parse_args()
    reader = PlateReader(a.bundle, device=a.device, conf=a.conf)
    for path in a.images:
        if a.crop:
            text, conf = reader.read_crop(path)
            plates = [{"text": text, "latin": reader.to_latin(text), "text_confidence": round(conf, 4)}]
        else:
            plates = reader.read(path)
            if a.save_vis:
                Path(a.save_vis).mkdir(parents=True, exist_ok=True)
                cv2.imwrite(str(Path(a.save_vis) / Path(path).name), reader.draw(path, plates))
        if a.json:
            print(json.dumps({"image": str(path), "plates": plates}, ensure_ascii=False))
        else:
            print(f"{path}: " + (", ".join(f"{p['text']} ({p['latin']}, {p['text_confidence']:.2f})" for p in plates) or "no plate"))


if __name__ == "__main__":
    main()

In [ ]:
# requirements, app README and optional ONNX exports, then rebuild the zip
(ARTIFACT_DIR / "requirements.txt").write_text(
    "torch\ntorchvision\nultralytics>=8.4.0\nopencv-python\nnumpy\npillow\n", encoding="utf-8")
(ARTIFACT_DIR / "APP_README.md").write_text(f"""# Using the licence plate reader in an app

Unzip `lpd_lpr_artifacts.zip` and install the requirements: `pip install -r lpd_lpr_artifacts/requirements.txt`.

## Python
```python
import sys; sys.path.insert(0, "lpd_lpr_artifacts")
from plate_reader import PlateReader
reader = PlateReader("lpd_lpr_artifacts", device="cpu")      # or "cuda" / "mps"
for p in reader.read("car.jpg"):                            # path, PIL image or cv2 BGR array
    print(p["text"], p["latin"], p["det_confidence"], p["text_confidence"], p["box"])
text, conf = reader.read_crop("plate_crop.jpg")             # recognizer only
```
`text` is the Persian plate string (`DD L DDD DD`, e.g. `12ب34567`; alef is written `الف`), `latin` a transliteration
(`12-B-345-67`). `text_confidence` is the lowest per-character probability, so reject reads below about 0.5.

## Command line
```
python lpd_lpr_artifacts/plate_reader.py car.jpg other.jpg --json --save-vis annotated/
python lpd_lpr_artifacts/plate_reader.py crop.jpg --crop
```

## Minimal web API (FastAPI)
```python
from fastapi import FastAPI, UploadFile
import cv2, numpy as np, sys
sys.path.insert(0, "lpd_lpr_artifacts")
from plate_reader import PlateReader
app, reader = FastAPI(), PlateReader("lpd_lpr_artifacts")

@app.post("/read")
async def read(file: UploadFile):
    img = cv2.imdecode(np.frombuffer(await file.read(), np.uint8), cv2.IMREAD_COLOR)
    return {{"plates": reader.read(img)}}
```

## Without PyTorch (ONNX, optional files)
* `lpd_best.onnx`: the detector (input 1x3x{LPD_IMGSZ}x{LPD_IMGSZ}, RGB 0-1, letterboxed). Easiest via `YOLO("lpd_best.onnx")`, or onnxruntime with your own pre/post-processing.
* `lpr_best.onnx`: the recognizer. Input `image` [N,3,64,256], the output of `lpr_models.preprocess` (letterbox + ImageNet normalization).
  Outputs `digit_logits` [N,7,10] and `letter_logits` [N,{N_LETTERS}]. Decode with `lpr_models.decode` or argmax + `lpr_config.json` vocabularies.

## Settings (lpr_config.json)
Detector confidence threshold {LPR_CONFIG['yolo_conf_threshold']}, detector image size {LPD_IMGSZ}, crop padding {LPR_CONFIG['crop_padding']}, recognizer: {BEST_LPR_NAME}.
""", encoding="utf-8")
for _f in ("plate_reader.py", "requirements.txt", "APP_README.md"):
    if _f not in BUNDLE_FILES:
        BUNDLE_FILES.append(_f)

# optional ONNX exports (for apps without PyTorch); failures are logged, never fatal
try:
    import onnx  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"], check=False)
try:
    torch.onnx.export(_best_cpu, torch.zeros(1, 3, *LPR_INPUT_HW), str(ARTIFACT_DIR / "lpr_best.onnx"),
                      input_names=["image"], output_names=["digit_logits", "letter_logits"],
                      dynamic_axes={"image": {0: "batch"}, "digit_logits": {0: "batch"}, "letter_logits": {0: "batch"}},
                      opset_version=17, dynamo=False)
    try:
        import onnxruntime as ort

        _sess = ort.InferenceSession(str(ARTIFACT_DIR / "lpr_best.onnx"), providers=["CPUExecutionProvider"])
        _o = _sess.run(None, {"image": _xs.numpy()})
        log(f"lpr_best.onnx OK: max |onnx - torch| = {max(np.abs(a - b.numpy()).max() for a, b in zip(_o, _eager)):.2e}")
    except ImportError:
        log("lpr_best.onnx exported (onnxruntime not installed, not cross-checked)")
    BUNDLE_FILES.append("lpr_best.onnx") if "lpr_best.onnx" not in BUNDLE_FILES else None
    shutil.copy2(ARTIFACT_DIR / "lpr_best.onnx", LPR_MODEL_DIRS[BEST_LPR_NAME] / "model.onnx")
except Exception as e:
    log("recognizer ONNX export skipped:", repr(e)[:300])
try:
    _onnx_path = Path(YOLO(str(ARTIFACT_DIR / "lpd_best.pt")).export(format="onnx", imgsz=LPD_IMGSZ, device="cpu"))
    if _onnx_path.resolve() != (ARTIFACT_DIR / "lpd_best.onnx").resolve():
        shutil.move(str(_onnx_path), ARTIFACT_DIR / "lpd_best.onnx")
    BUNDLE_FILES.append("lpd_best.onnx") if "lpd_best.onnx" not in BUNDLE_FILES else None
    log("lpd_best.onnx exported")
    shutil.copy2(ARTIFACT_DIR / "lpd_best.onnx", LPD_MODEL_DIR / "best.onnx")
except Exception as e:
    log("detector ONNX export skipped:", repr(e)[:300])
write_bundle_zip()

### Bundle verification (fresh scope)

`verify_bundle` uses **only the zip file** and raw LPD test images. It imports everything itself and reads no notebook variables. It:

1. unzips the bundle to a temporary folder;
2. loads `lpd_best.pt` and `lpr_best.ts`, plus `lpr_models.py` imported from the bundle;
3. runs YOLO on 3 LPD **test** images and crops the most confident box with the recommended padding;
4. recognizes the crop and prints the predicted strings next to the ground truth.

If the detector finds no plate, which should only happen with the 1-epoch smoke-test model, it crops the ground-truth box instead and labels the output `[GT box]`. The recognizer part of the bundle is therefore always tested.

In [ ]:
def verify_bundle(zip_path, lpd_image_dir, plate_csv=None, lpd_label_dir=None, n=3):
    import importlib.util
    import json
    import tempfile
    import zipfile
    from pathlib import Path

    import cv2
    import matplotlib.pyplot as plt
    import pandas as pd
    import torch
    from ultralytics import YOLO

    tmp = Path(tempfile.mkdtemp(prefix="lpd_lpr_bundle_"))
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(tmp)
    root = tmp / "lpd_lpr_artifacts"
    spec = importlib.util.spec_from_file_location("bundle_lpr_models", root / "lpr_models.py")
    M = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(M)
    cfg = json.loads((root / "lpr_config.json").read_text(encoding="utf-8"))
    det = YOLO(str(root / "lpd_best.pt"))
    rec = torch.jit.load(str(root / "lpr_best.ts"), map_location="cpu").eval()
    gt = {}
    if plate_csv is not None and Path(plate_csv).is_file():
        gt = {r.image_path: r.label for r in pd.read_csv(plate_csv, dtype=str, keep_default_na=False).itertuples()}
    test_files = (root / "lpd_test_split.txt").read_text().split()[:n]
    fig, axes = plt.subplots(2, n, figsize=(8 * n, 11), gridspec_kw={"height_ratios": [4, 1]})
    for k, fname in enumerate(test_files):
        img = cv2.imread(str(Path(lpd_image_dir) / fname))
        res = det.predict(img, conf=cfg["yolo_conf_threshold"], imgsz=cfg["yolo_imgsz"], device="cpu", verbose=False)[0]
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        text, latin, xyxy, src = "<no plate detected>", "-", None, "YOLO"
        if len(res.boxes):
            xyxy = res.boxes.xyxy[int(res.boxes.conf.argmax())].tolist()
        elif lpd_label_dir is not None and (Path(lpd_label_dir) / (Path(fname).stem + ".txt")).is_file():
            # Detector found nothing (expected only for the 1-epoch smoke model): crop the ground-truth box
            # instead so the crop -> recognizer path of the bundle is still exercised.
            _, cx, cy, bw, bh = map(float, (Path(lpd_label_dir) / (Path(fname).stem + ".txt")).read_text().split()[:5])
            H, W = img.shape[:2]
            xyxy, src = [(cx - bw / 2) * W, (cy - bh / 2) * H, (cx + bw / 2) * W, (cy + bh / 2) * H], "GT box (no detection)"
        if xyxy is not None:
            crop = M.crop_with_padding(img, xyxy, cfg["crop_padding"])
            with torch.no_grad():
                out = rec(M.preprocess(crop, bgr=True)[None])
            text = M.decode(out, cfg["letter_vocab"])
            idx = [cfg["letter_vocab"].index(text[2:-5])] if len(text) >= 8 else []
            latin = f"{text[:2]}-{cfg['letter_vocab_latin'][idx[0]] if idx else '?'}-{text[-5:-2]}-{text[-2:]}"
            x1, y1, x2, y2 = map(int, xyxy)
            cv2.rectangle(rgb, (x1, y1), (x2, y2), (255, 0, 0), max(2, rgb.shape[1] // 300))
            axes[1, k].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        print(f"{fname}: [{src}] predicted {text!r} ({latin}), conf {float(res.boxes.conf.max()) if len(res.boxes) else 0:.2f}; "
              f"ground truth {[M.normalize_label(t) for t in gt.get(fname, '?').split()]}")
        axes[0, k].imshow(rgb)
        axes[0, k].set_title(f"{fname}: {latin}\n[{src}]")
        for a in axes[:, k]:
            a.axis("off")
    plt.tight_layout()
    show("bundle_verification")
    return tmp


_bundle_tmp = verify_bundle(OUT_DIR / "lpd_lpr_artifacts.zip", LPD_DIR / "images", LPD_DIR / "plate_labels.csv", LPD_DIR / "labels")

# the app package, run as a separate process from the unzipped bundle (proves it needs nothing from this notebook)
_reader = _bundle_tmp / "lpd_lpr_artifacts" / "plate_reader.py"
_full = [str(LPD_DIR / "images" / f) for f in (ARTIFACT_DIR / "lpd_test_split.txt").read_text().split()[:2]]
_crop = [str(LPR_DIR / "detections" / f) for f in LPR_SPLIT_DFS["test"].filename[:2]]
for _args in (_full + ["--json", "--device", "cpu"], _crop + ["--crop", "--json", "--device", "cpu"]):
    _r = subprocess.run([sys.executable, str(_reader), *_args], capture_output=True, text=True, cwd=str(_bundle_tmp))
    log("plate_reader.py", "(crops)" if "--crop" in _args else "(full images)", "exit", _r.returncode)
    print(_r.stdout.strip() or _r.stderr[-2000:])
    assert _r.returncode == 0, _r.stderr[-2000:]
print("ground truth of the crops:", LPR_SPLIT_DFS["test"].text[:2].tolist())
shutil.rmtree(_bundle_tmp, ignore_errors=True)

### Result files written by this run

The zip is rewritten here so it also contains the verification figure and the complete run log.

In [ ]:
log("===== run finished")
write_bundle_zip()
_res = pd.DataFrame([{"file": f.relative_to(RESULTS_DIR).as_posix(), "kB": round(f.stat().st_size / 1e3, 1)}
                     for f in sorted(RESULTS_DIR.rglob("*")) if f.is_file()])
with pd.option_context("display.max_rows", 300):
    display(_res)
print(f"figures, CSVs and the log are in {RESULTS_DIR} and inside {BUNDLE_ZIP.name} under lpd_lpr_artifacts/outputs/")
print(f"models + app package: {ARTIFACT_DIR} (and in the zip)")
print("\none output folder per trained model:")
for _d in sorted(MODELS_DIR.iterdir()):
    print(f"  {_d}/")
    for _f in sorted(_d.iterdir()):
        print(f"      {_f.name:28s} {_f.stat().st_size / 1e6:8.2f} MB")

## 3. End-to-End License Plate Detection & Recognition

In [ ]:
# TODO: Load the trained models for both LPD and LPR

In [ ]:
# TODO: Define an end-to-end class for license plate detection and recognition

In [ ]:
# TODO: Plot some end-to-end prediction on test data